# 评测代码精读：从“怎么写”到“怎么想”

`notebooks/eval_tutorial.ipynb` 讲的是**评测体系怎么用**（题目长什么样、判分规则、怎么跑、结果怎么读）。这本 notebook 换一个角度，面向想提升工程能力的读者：

1. **逐个文件精读 `evals/` 的代码**：每个函数为什么这样写、用了哪些常见的工程手法、有哪些取舍
2. **带着问题重读真实评测结果**（glm-5.3-flash）：不是复述数字，而是演示“看到一个数字 → 提出疑问 → 找数据验证 → 得出结论”的完整过程
3. **把精读中发现的问题列出来**：有些是评测代码本身的缺陷，有些是对结果的解读需要打折扣的地方。每个问题都给出可以亲手运行的复现

**读法建议**：遇到 “🤔 先想一想” 时，先停下来自己想（最好写下来），再展开下面的“分析”。这些问题没有标准答案，重要的是推理过程。

| 节 | 内容 | 需要的服务 |
| --- | --- | --- |
| 0 | 准备：一个不导入模块也能看源码的小工具 | 无 |
| 1 | 全景：评测在回答什么问题，代码怎么分层 | 无 |
| 2 | 评测集：数据就是测试用例 | 无 |
| 3 | `dataset.py`：不连库的静态检查 | 无 |
| 4 | `check_dataset.py`：评测题本身也要被测试 | MySQL（可选） |
| 5 | `compare.py`：判分器（**重点**） | 无 |
| 6 | `scoring.py`：归因、汇总、基线门禁 | 无 |
| 7 | `runner.py`：把一切串起来 | 无（只读代码） |
| 8 | `llm_cache.py`：录制回放与可复现性 | 无 |
| 9 | `multiturn.py`：多轮评测 | 无（只读代码） |
| 10 | 带着问题重读真实结果 | MySQL（10.5 可选） |
| 11 | 问题清单与练习 | 无 |

运行前提：`uv sync --group notebook`。第 4 节和 10.5 节要连数仓（`docker compose -f docker/docker-compose.yaml up -d mysql` 并写入数据），连不上时会跳过并给出说明，不影响其余部分。本 notebook **不需要 LLM key**，也不导入 `app.conf.app_config`。

## 0. 准备

In [1]:
import ast
import json
import math
import os
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)


def show(path: str, name: str | None = None) -> None:
    # 用 ast 从源文件里取出某个函数 / 类的源码并显示，不需要 import 这个模块
    # （runner.py 一导入就会加载项目配置、编译问数图，需要 LLM key；只读代码时没必要）
    source = (ROOT / path).read_text(encoding="utf-8")
    if name is None:
        snippet = source
    else:
        tree = ast.parse(source)
        node = next(
            n
            for n in ast.walk(tree)
            if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))
            and n.name == name
        )
        snippet = ast.get_source_segment(source, node)
    display(
        Markdown(
            f"`{path}`"
            + (f" · `{name}`" if name else "")
            + f"\n```python\n{snippet}\n```"
        )
    )


show("evals/compare.py", "_decimals")

`evals/compare.py` · `_decimals`
```python
def _decimals(x: float) -> int:
    """预测值的小数位数，用来判断它是否是标准值四舍五入后的结果"""
    text = f"{x:.10f}".rstrip("0")
    return len(text.split(".")[1]) if "." in text else 0
```

> 小工具 `show()` 本身就是一个工程习惯：**文档里的代码直接从源文件取**，而不是复制粘贴。代码改了，教程自动跟着变，不会出现“文档里的代码和仓库里对不上”。

## 1. 全景：评测在回答什么问题

一个 NL2SQL Agent 的评测，本质上要回答四个问题：

| 问题 | 指标 | 谁负责 |
| --- | --- | --- |
| 答对了吗 | 执行准确率：模型 SQL 的执行结果与标准 SQL 的执行结果“业务上等价”的比例 | `compare.py` |
| 错在哪 | 失败环节：流程异常 → （意图错误）→ 召回遗漏 → 过滤误删 → SQL 不可执行 → 结果不符 | `scoring.py` |
| 代价多大 | 每题 LLM 调用次数、tokens、耗时、修正次数 | `runner.py` 的 `UsageCallback` |
| 变差了吗 | 与基线比较：必过题失败、准确率下降超过阈值、调用次数上升超过阈值 | `scoring.compare_with_baseline` |

再加一个经常被忽略的第五个问题：**评测本身可信吗？** 题目有没有区分度、判分器会不会误判、结果能不能复现。`evals/` 里有相当一部分代码是在回答这个问题（`dataset.py`、`check_dataset.py`、`--predictor gold/trap`、`llm_cache.py`）。

### 代码分层：纯函数核心 + 有副作用的外壳

把文件按“是否碰数据库 / LLM”分开看：

In [2]:
files = {
    "evals/compare.py": "结果等价判定",
    "evals/scoring.py": "归因、汇总、基线比较、渲染报告",
    "evals/dataset.py": "加载评测集、静态检查、泄漏检查",
    "evals/llm_cache.py": "LLM 调用录制回放",
    "evals/check_dataset.py": "在数仓上校验评测题",
    "evals/runner.py": "命令行入口：单轮答题、打分、写报告",
    "evals/multiturn.py": "多轮答题",
}
rows = []
for path, role in files.items():
    source = (ROOT / path).read_text(encoding="utf-8")
    tree = ast.parse(source)
    imports = {
        ".".join(n.module.split(".")[:2])
        for n in ast.walk(tree)
        if isinstance(n, ast.ImportFrom) and n.module and n.module.startswith("app")
    }
    io = any(k in source for k in ("session_factory", "astream", "sqlite3.connect"))
    rows.append(
        {
            "文件": path,
            "职责": role,
            "行数": source.count("\n"),
            "依赖的项目模块": ", ".join(sorted(imports)) or "-",
            "有 IO（库 / LLM / 文件）": "是" if io else "否",
        }
    )
pd.DataFrame(rows)

,文件,职责,行数,依赖的项目模块,有 IO（库 / LLM / 文件）
0,evals/compare.py,结果等价判定,223,-,否
1,evals/scoring.py,归因、汇总、基线比较、渲染报告,461,-,否
2,evals/dataset.py,加载评测集、静态检查、泄漏检查,185,"app.conf, app.core",否
3,evals/llm_cache.py,LLM 调用录制回放,70,-,是
4,evals/check_dataset.py,在数仓上校验评测题,178,app.clients,是
5,evals/runner.py,命令行入口：单轮答题、打分、写报告,407,"app.agent, app.clients, app.conf, app.core, app.services",是
6,evals/multiturn.py,多轮答题,259,"app.agent, app.clients, app.conf, app.core, app.services",是


这是一种很常见、也很值得模仿的结构，叫 **functional core, imperative shell**（纯函数核心、命令式外壳）：

- **核心**（`compare.py`、`scoring.py`、`dataset.py` 的大部分）只接收普通的 Python 数据（`list[dict]`、dataclass），返回普通的数据，不连库、不调模型、不读配置。所以它们可以被 `tests/test_compare.py`、`tests/test_scoring.py`、`tests/test_eval_dataset.py` 在毫秒级、无任何外部服务的情况下测试
- **外壳**（`runner.py`、`multiturn.py`、`check_dataset.py`）负责连库、调 Agent、并发、写文件，逻辑尽量薄，把“判断”交给核心

> **为什么这很重要**：评测代码一旦有 bug，影响的是你对整个系统的判断（“改进了 3 个点”可能是判分器的 bug）。判分逻辑必须是最容易测试、测试最充分的部分。
>
> `llm_cache.py` 是个有趣的例外：它有 IO（sqlite），但 IO 被封装在一个类里、通过路径注入，`tests/test_scoring.py::test_llm_cache_roundtrip` 用 pytest 的 `tmp_path` 就能测。“可测试”不等于“没有 IO”，而是“IO 可以被替换”。

### 一次单轮评测的数据流

```
traps.yaml ──load_dataset──► items
                               │
                    ┌──────────┴───────────┐
          (串行) 执行全部 gold_sql      为每题 × 每种问法建任务
          gold_cache[id] = rows                │
                    │            asyncio.gather + Semaphore(4)
                    │                          ▼
                    │       evaluate_case(item, variant, question, predictor)
                    │         ├─ predictor=agent: run_agent → 问数图 astream
                    │         │     捕获每个节点的输出 outputs[node]
                    │         │     取最终 SQL、召回/过滤到的表和指标
                    │         ├─ predictor=gold / trap: 直接用标准 SQL / 错误写法
                    │         ├─ execute(sql) ──► pred_rows
                    └────────►├─ results_equal(pred_rows, gold_rows)   ← compare.py
                              └─ attribute_failure(...)               ← scoring.py
                                       │
                         list[CaseResult] ──summarize──► summary
                                       │                     │
                       results.jsonl / summary.json / summary.md
                                       │
                          compare_with_baseline ──► 退出码 0 / 1
```

下面按“数据 → 检查 → 判分 → 汇总 → 编排”的顺序读代码。

## 2. 评测集：数据就是测试用例

评测集是 YAML，不是代码。但它的设计决定了评测能测出什么，比任何一行 Python 都重要。

In [3]:
from evals.dataset import DEFAULT_DATASET, MULTITURN_DATASET, load_dataset

traps = load_dataset(DEFAULT_DATASET)
multiturn = load_dataset(MULTITURN_DATASET)
item = next(i for i in traps if i["id"] == "fanout_04")
print(yaml.dump(item, allow_unicode=True, sort_keys=False))

id: fanout_04
category: fanout
tags:
- 多表
- 分组
- 销量
difficulty: medium
today: '2025-12-31'
turns:
- question: 2025 年各品牌卖出了多少件商品？
  paraphrases:
  - 今年每个品牌的销量是多少件
  gold_sql: "select k.brand_name as 品牌, sum(i.quantity) as 销量\nfrom fact_order_item\
    \ i\njoin fact_order o on i.order_id = o.order_id\njoin dim_sku k on i.sku_id\
    \ = k.sku_id\nwhere o.pay_time is not null\n  and o.pay_date_id between 20250101\
    \ and 20251231\ngroup by k.brand_name\n"
  trap_sql: "select k.brand_name as 品牌, sum(o.item_quantity) as 销量\nfrom fact_order\
    \ o\njoin fact_order_item i on i.order_id = o.order_id\njoin dim_sku k on i.sku_id\
    \ = k.sku_id\nwhere o.pay_time is not null\n  and o.pay_date_id between 20250101\
    \ and 20251231\ngroup by k.brand_name\n"
  trap: 用订单级的总件数按明细关联后求和，件数被重复计算且被算到所有品牌上
  expect:
    tables:
    - fact_order
    - fact_order_item
    - dim_sku
  order_matters: false



### 每个字段背后的设计意图

| 字段 | 如果没有它会怎样 |
| --- | --- |
| `gold_sql`（而不是标准答案的数值） | 数据一改，答案就要手工重算；有了 SQL，标准答案每次评测时现算，数据生成器改了也不用改题 |
| `trap_sql` + `trap` | 无法证明这道题“能测出它声称的陷阱”。比如一道“扇出”题，如果数据里恰好没有一对多的情况，扇出写法和正确写法结果一样，模型写错了也会被判对。有了 `trap_sql`，`check_dataset` 会证明错误写法确实得到不同的结果（**可证伪性**） |
| `expect` | 答错时只知道“错了”，不知道是召回没找到表、过滤删掉了表，还是 SQL 写错了 |
| `paraphrases` | 只能测“标准问法”，测不出模型对措辞是否敏感。真实用户不会照着题库提问 |
| `today` | “上个月”“今年”这类相对时间的题，结果会随运行日期变化，无法复现 |
| `order_matters` | 要么所有题都比较行顺序（“各品牌销量”没要求排序却被判错），要么都不比较（“前 3 名”顺序错了也被判对） |
| `must_pass` | 所有题一视同仁；核心场景（“今年销售额是多少”）退化时不会被特别拦下 |

> **工程启示：把“这条数据为什么这样”写成字段。** 一道题的 `trap` 是一句自然语言说明，它不参与任何计算，却让半年后的你（或同事）知道这道题在测什么。`traps.yaml` 文件头那段字段说明也是同样的道理：数据格式本身就是接口，接口要有文档。

In [4]:
def flat_turns(items):
    for it in items:
        for t, turn in enumerate(it["turns"]):
            yield it, t, turn


pd.DataFrame(
    [
        {
            "评测集": name,
            "题 / 对话": len(items),
            "轮次": sum(len(i["turns"]) for i in items),
            "类别": len({i["category"] for i in items}),
            "有换问法的轮次": sum(
                bool(turn.get("paraphrases")) for _, _, turn in flat_turns(items)
            ),
            "必过": sum(bool(i.get("must_pass")) for i in items),
            "order_matters": sum(
                bool(turn.get("order_matters")) for _, _, turn in flat_turns(items)
            ),
        }
        for name, items in {"traps.yaml": traps, "multiturn.yaml": multiturn}.items()
    ]
)

,评测集,题 / 对话,轮次,类别,有换问法的轮次,必过,order_matters
0,traps.yaml,50,50,10,50,9,2
1,multiturn.yaml,22,48,7,0,6,3


🤔 **先想一想**：单轮 50 题、每题 1 种换问法。准确率每变化 1 道题是多少个百分点？如果你改了一个 Prompt，分数从 96% 变成 94%，你会认为“变差了”吗？

<details><summary>分析</summary>

原问法 50 题，1 题 = 2 个百分点；加上换问法 100 次作答，1 次 = 1 个百分点。96% → 94% 就是 1 道题的差别。

LLM 即使 `temperature=0` 也不是完全确定的（第 8 节会看到，向量计算的批次噪声都能让 2.6% 的提示词变化），1 道题的波动完全可能是噪声。**评测集越小，能可靠检测到的改进就越大**。第 10.2 节会用置信区间把这个直觉量化。

</details>

## 3. `dataset.py`：不连库的静态检查

`dataset.py` 做所有“不需要数据库就能发现”的问题检查，并且被 `tests/test_eval_dataset.py` 调用，也就是说**每次 `uv run pytest` 都会检查评测集**。

In [5]:
show("evals/dataset.py", "check_structure")

`evals/dataset.py` · `check_structure`
```python
def check_structure(items: list[dict], per_category: int | None) -> list[str]:
    """检查字段完整性 id 唯一性 各类别题数 以及 expect 与 SQL 的一致性"""
    errors: list[str] = []
    ids = Counter(item.get("id") for item in items)
    errors += [f"id 重复：{i}" for i, n in ids.items() if n > 1]

    if per_category is not None:
        for category, n in Counter(item.get("category") for item in items).items():
            if n != per_category:
                errors.append(f"类别 {category} 有 {n} 题，预期 {per_category} 题")

    metrics = metric_names()
    for item in items:
        prefix = item.get("id", "<缺少 id>")
        for key in ("id", "category", "difficulty", "today", "turns"):
            if key not in item:
                errors.append(f"{prefix}: 缺少字段 {key}")
        for t, turn in enumerate(item.get("turns", [])):
            where = f"{prefix} 第 {t + 1} 轮"
            for key in REQUIRED_TURN_FIELDS:
                if key not in turn:
                    errors.append(f"{where}: 缺少字段 {key}")
            errors += _check_intent(turn, t, where)
            if "gold_sql" not in turn:
                continue
            expect = turn.get("expect") or {}
            missing = set(expect.get("tables", [])) - tables_in_sql(turn["gold_sql"])
            if missing:
                errors.append(
                    f"{where}: expect.tables 中的 {sorted(missing)} 没有出现在标准 SQL 中"
                )
            unknown = set(expect.get("metrics", [])) - metrics
            if unknown:
                errors.append(
                    f"{where}: expect.metrics 中的 {sorted(unknown)} 不是已定义的指标"
                )
            for key in ("gold_sql", "trap_sql"):
                try:
                    ensure_readonly(turn.get(key, ""))
                except ValueError as e:
                    errors.append(f"{where}: {key} 未通过只读校验：{e}")
    return errors
```

### 读这段代码时值得注意的手法

1. **收集所有错误，而不是遇到第一个就抛异常**：`errors: list[str]`，最后一起返回。改评测集时往往一次改很多题，一次看到全部问题比“改一个、跑一次、再看下一个”高效得多。`check_dataset.py` 也是同样的风格
2. **错误信息带定位**：`where = f"{prefix} 第 {t + 1} 轮"`。好的错误信息 = 在哪里 + 什么错了 + （最好）该怎么改
3. **交叉校验两个数据源**：`expect.tables` 必须出现在 `gold_sql` 解析出的表里（用 sqlglot 解析 SQL，而不是字符串搜索）；`expect.metrics` 必须是 `meta_config.yaml` 里定义过的指标。两份人工维护的数据之间的一致性，最适合交给程序检查
4. **复用生产代码做校验**：`ensure_readonly` 就是 Agent 执行 SQL 前用的那个只读检查。评测题能通过生产环境的安全检查，才说明它在生产链路上是可执行的

### 泄漏检查：few-shot 示例不能“偷看”考题

In [6]:
show("evals/dataset.py", "check_leakage")

`evals/dataset.py` · `check_leakage`
```python
def check_leakage(items: list[dict], examples: list[dict]) -> list[str]:
    """
    检查 few-shot 示例是否与评测题过于相似

    示例会在问数时作为参考写法召回，如果和评测题几乎一样，评测分数就会虚高；
    问题（含 paraphrases）和 SQL 都要比较
    """
    errors: list[str] = []
    for item in items:
        for turn in item.get("turns", []):
            questions = [
                turn.get("question", ""),
                *turn.get("paraphrases", []),
                *([turn["gold_standalone"]] if turn.get("gold_standalone") else []),
            ]
            for i, example in enumerate(examples, start=1):
                for q in questions:
                    ratio = SequenceMatcher(
                        None, _norm(q), _norm(example["question"])
                    ).ratio()
                    if ratio >= LEAKAGE_THRESHOLD:
                        errors.append(
                            f"{item['id']}: 问题与第 {i} 条示例过于相似（{ratio:.2f}）：{q} / {example['question']}"
                        )
                sql_ratio = SequenceMatcher(
                    None, _norm(turn.get("gold_sql", "")), _norm(example["sql"])
                ).ratio()
                if sql_ratio >= SQL_LEAKAGE_THRESHOLD:
                    errors.append(
                        f"{item['id']}: 标准 SQL 与第 {i} 条示例 SQL 过于相似（{sql_ratio:.2f}）"
                    )
    return errors
```

In [7]:
from difflib import SequenceMatcher

from evals.dataset import _norm

pairs = [
    ("2025 年华北大区的销售额是多少？", "2024 年华北大区的销售额是多少？"),
    ("2025 年华北大区的销售额是多少？", "2024年华东的销售额是多少"),
    ("2025 年各品牌卖出了多少件商品？", "2024 年各类目卖出了多少件商品？"),
    ("2025 年各品牌卖出了多少件商品？", "去年每个店铺的退款金额"),
]
pd.DataFrame(
    [
        {
            "评测题": a,
            "示例": b,
            "相似度": round(SequenceMatcher(None, _norm(a), _norm(b)).ratio(), 2),
        }
        for a, b in pairs
    ]
)

,评测题,示例,相似度
0,2025 年华北大区的销售额是多少？,2024 年华北大区的销售额是多少？,0.94
1,2025 年华北大区的销售额是多少？,2024年华东的销售额是多少,0.80
2,2025 年各品牌卖出了多少件商品？,2024 年各类目卖出了多少件商品？,0.81
3,2025 年各品牌卖出了多少件商品？,去年每个店铺的退款金额,0.07


🤔 **先想一想**：阈值是 0.8。上面第 1 对相似度很高会被拦下；第 3 对“各品牌 / 各类目卖出了多少件”呢？它在“写法模式”上几乎一样（明细关联订单、按支付口径计件），却可能逃过检查。文本相似度能防住什么、防不住什么？

<details><summary>分析</summary>

`SequenceMatcher` 比较的是字符序列，它能拦住“换个年份、换个地名”的复制粘贴，但拦不住“换个维度、同样的坑”。而对 NL2SQL 来说，真正起作用的恰恰是写法模式：示例里演示了“计件要关联订单表加支付过滤”，模型在评测题上就会照做。

这就是 CLAUDE.md 里那句“示例与评测题的相似度检查是文本层面的，不能完全排除写法模式层面的提示”的含义，也是为什么要有 `--no-examples` 消融：**静态检查防住明显的泄漏，消融实验度量剩下的影响**。两种手段互补。

</details>

## 4. `check_dataset.py`：评测题本身也要被测试

如果说 `dataset.py` 检查“格式”，`check_dataset.py` 检查的是“**这道题在这份数据上有没有意义**”。核心在 `check_results`：

In [8]:
show("evals/check_dataset.py", "check_results")

`evals/check_dataset.py` · `check_results`
```python
async def check_results(items: list[dict]) -> tuple[list[str], list[tuple]]:
    """在数仓上执行标准 SQL 与错误写法，确认前者有结果、后者结果不同"""
    errors: list[str] = []
    report: list[tuple] = []
    dw_mysql_client_manager.init()
    try:
        async with dw_mysql_client_manager.session_factory() as session:
            for item in items:
                for t, turn in enumerate(item["turns"]):
                    where = f"{item['id']} 第 {t + 1} 轮"
                    try:
                        gold = await run_sql(session, turn["gold_sql"])
                    except Exception as e:
                        errors.append(f"{where}: 标准 SQL 执行失败：{e}")
                        await session.rollback()
                        continue
                    try:
                        trap = await run_sql(session, turn["trap_sql"])
                    except Exception as e:
                        errors.append(f"{where}: 错误写法执行失败：{e}")
                        await session.rollback()
                        continue

                    order = bool(turn.get("order_matters"))
                    if not gold or all(v is None for row in gold for v in row.values()):
                        errors.append(f"{where}: 标准 SQL 结果为空")
                    # 结果追问轮的关键词必须出现在标准答案里，否则直接回答永远判不对
                    cells = {str(v) for row in gold for v in row.values()}
                    for text in turn.get("gold_answer_contains") or []:
                        if not any(text in cell for cell in cells):
                            errors.append(
                                f"{where}: gold_answer_contains 的“{text}”不在标准结果里"
                            )
                    if not results_equal(gold, gold, order):
                        errors.append(f"{where}: 判定器认为标准结果与自身不同")
                    trap_same = results_equal(trap, gold, order)
                    if trap_same:
                        errors.append(
                            f"{where}: 错误写法的结果与标准答案相同，测不出该陷阱"
                        )
                    elif results_equal(_rounded(trap), gold, order):
                        # 判定器接受四舍五入后的预测值，模型若对错误结果取两位小数就会被判对
                        errors.append(
                            f"{where}: 错误写法保留两位小数后与标准答案无法区分，区分度太弱"
                        )
                    report.append(
                        (
                            item["id"]
                            if len(item["turns"]) == 1
                            else f"{item['id']}#{t + 1}",
                            len(gold),
                            _preview(gold),
                            _preview(trap),
                            not trap_same,
                        )
                    )
    finally:
        await dw_mysql_client_manager.close()
    return errors, report
```

逐条看它在防什么：

| 检查 | 防的是什么 |
| --- | --- |
| 标准 SQL 执行失败 / 结果为空 | 题目本身是坏的；或者数据生成器改了，这道题已经没有数据了。**空结果特别危险**：模型随便写一条查不到数据的 SQL 也能“答对” |
| `gold_answer_contains` 不在标准结果里 | 直接回答的轮次永远判不对 |
| `results_equal(gold, gold)` 为假 | **判分器的自反性**：一个结果和自己比都不相等，说明判分器有 bug（例如 NULL、浮点处理） |
| 错误写法结果与标准答案相同 | 这道题测不出它声称的陷阱（可证伪性） |
| 错误写法保留两位小数后与标准答案无法区分 | 判分器允许四舍五入，如果陷阱只让结果差 0.001，模型对错误结果四舍五入后也会被判对，**区分度太弱** |

最后一条特别值得体会：**它把判分器的宽松规则和评测题的有效性联系起来了**。判分器每放宽一条规则，都可能让某些题失去区分度，所以 CLAUDE.md 要求“放宽判分规则后必须重跑 `check_dataset` 和 `--predictor trap`”。

### 测试评测器：gold 100% / trap 0%

`runner.py --predictor gold` 用标准 SQL 作答，应该 100% 通过；`--predictor trap` 用错误写法作答，应该 0% 通过。这是对**判分器**的两个方向的测试：

- gold 不到 100% → 判分器**误杀**（false negative），比如没有处理好 Decimal 和 float
- trap 不到 0% → 判分器**误放**（false positive），比如容差太大

🤔 **先想一想**：trap 0% 能证明判分器不会误放吗？

<details><summary>分析</summary>

不能。它只证明判分器能识别**我们预先写好的那一种**错误写法。模型的错误千奇百怪，判分器可能对另一种错误写法误放。第 5.4 节会构造几个 trap 自检覆盖不到、但判分器确实会误放的例子。

这和软件测试的一般道理一样：测试只能证明 bug 存在，不能证明 bug 不存在。自检是必要条件，不是充分条件。

</details>

下面这个单元格会连数仓真的跑一遍（只检查单轮评测集，约 10 秒）；连不上数据库时会跳过。

In [9]:
import subprocess

proc = subprocess.run(
    [sys.executable, "-m", "evals.check_dataset"],
    capture_output=True,
    text=True,
    env={**os.environ, "LLM_API_KEY": os.environ.get("LLM_API_KEY", "dummy")},
    timeout=300,
)
out = proc.stdout + proc.stderr
if "Can't connect" in out or "2003" in out:
    print("连不上 MySQL，跳过（先启动 docker compose 里的 mysql 并写入数仓数据）")
else:
    lines = out.strip().splitlines()
    print("\n".join(lines[:8] + ["……"] + lines[-6:]))

few-shot 示例 10 条，已在数仓上执行
题目                         行数  标准答案                                错误写法                                可区分
order_vs_pay_01             1  46432548.78                         50384235.76                         是
order_vs_pay_02            12  12 行                                12 行                                是
order_vs_pay_03             1  4541574.33, 4219845.12              4541574.33, 4541574.33              是
order_vs_pay_04             4  4 行                                 4 行                                 是
order_vs_pay_05             1  2573, 3933731.55                    2578, 3951686.98                    是
fanout_01                   7  7 行                                 7 行                                 是
……
funnel_03                   1  0.5549                              0.2082                              是
funnel_04                   1  0.5117                              0.5352                              是
funnel_05                

## 5. `compare.py`：判分器（重点）

判分器决定了“答对”的定义。它要在两种错误之间取平衡：

- **太严**：模型输出 `'2025-01'`、标准答案输出 `1`，业务上一样却被判错 → 低估模型，还会让你去“修”根本没错的东西
- **太松**：模型算错了一个比率，却因为容差被判对 → 高估模型，问题被掩盖

Spider、BIRD 这些学术基准用的是“结果集完全相同”（顺序无关）。这里参照 Spider 2.0 的宽松口径，规则写在模块 docstring 里：

In [10]:
import evals.compare as compare

print(compare.__doc__)


查询结果等价判定

判断模型生成的 SQL 与标准 SQL 的执行结果是否“业务上相同”，规则参照 Spider 2.0 的宽松口径：
1. 忽略列名和列顺序：标准答案的每一列都要在预测结果里找到一列内容相同的列，预测结果允许多出列
2. 行：order_matters 为 False 时按多重集合比较，为 True 时按顺序比较
3. 数值有容差：相对误差 1e-9 以内视为相等（只用来吸收浮点误差；金额可达数百万，更宽的容差会放过几元甚至几百元的差异）；预测值做了四舍五入（保留 1~4 位小数，
   或金额较大时取整）也视为相等；同一列允许整体是标准答案的 100 倍（比率 vs 百分数）
4. 日期统一转成 ISO 字符串，Decimal 和整数统一转成 float，字符串去掉首尾空白
5. 时间标签换算：一列预测值整体是“年月”标签（'2025-01'、202501、'2025年1月'）且只涉及一个年份时，
   可以按月份数字与标准列比较；日期 '2025-12-25' 与日期键 20251225 互相认可。
   标准 SQL 常按 dim_date.month 输出 1~12，模型常输出带年份的月份，两者业务上相同

本模块是纯函数，不依赖数据库和项目配置，方便单元测试



### 5.1 第一层：单元格怎么比

`normalize` 把各种类型统一成可比较的形式；`values_equal` 判断两个单元格是否相等。

In [11]:
show("evals/compare.py", "normalize")
show("evals/compare.py", "values_equal")

`evals/compare.py` · `normalize`
```python
def normalize(value: Any) -> Any:
    """把一个单元格的值转换成便于比较的形式"""
    if value is None:
        return None
    if isinstance(value, bool):
        return float(value)
    if isinstance(value, (int, float, Decimal)):
        return float(value)
    if isinstance(value, datetime):
        return value.isoformat(sep=" ")
    if isinstance(value, date):
        return value.isoformat()
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    text = str(value).strip()
    # 数据库驱动有时把 DECIMAL 以字符串返回，能转成数字的统一按数字比较
    try:
        return float(text)
    except ValueError:
        return text
```

`evals/compare.py` · `values_equal`
```python
def values_equal(pred: Any, gold: Any, scale: int = 1) -> bool:
    """判断单个预测值与标准值是否相等，scale 表示预测列相对标准列的倍数"""
    if pred is None or gold is None:
        return pred is None and gold is None
    if isinstance(pred, float) and isinstance(gold, float):
        expected = gold * scale
        if abs(pred - expected) <= REL_TOL * max(abs(expected), 1e-9):
            return True
        if abs(pred - expected) < 1e-9:
            return True
        # 预测值做了四舍五入：把标准值按同样的小数位数取整后比较
        digits = _decimals(pred)
        if 1 <= digits <= 4 or (digits == 0 and abs(expected) >= 1000):
            return round(expected, digits) == round(pred, digits)
        return False
    return pred == gold
```

In [12]:
from evals.compare import normalize, values_equal

cases = [
    (
        "Decimal 与 float",
        normalize(__import__("decimal").Decimal("12.50")),
        normalize(12.5),
        1,
    ),
    ("字符串形式的数字", normalize("12.50"), normalize(12.5), 1),
    ("四舍五入到 2 位", 0.31, normalize(0.3141592), 1),
    ("四舍五入到 2 位，但舍错了", 0.32, normalize(0.3141592), 1),
    ("百分数（scale=100）", 31.42, normalize(0.3141592), 100),
    ("大金额取整", 14577501.0, 14577500.76, 1),
    ("小数值取整（<1000 不允许）", 3.0, 3.4, 1),
]
pd.DataFrame(
    [
        {"情形": n, "预测": p, "标准": g, "倍数": s, "相等": values_equal(p, g, s)}
        for n, p, g, s in cases
    ]
)

,情形,预测,标准,倍数,相等
0,Decimal 与 float,12.50,1.250000e+01,1,True
1,字符串形式的数字,12.50,1.250000e+01,1,True
2,四舍五入到 2 位,0.31,3.141592e-01,1,True
3,四舍五入到 2 位，但舍错了,0.32,3.141592e-01,1,False
4,百分数（scale=100）,31.42,3.141592e-01,100,True
5,大金额取整,14577501.00,1.457750e+07,1,True
6,小数值取整（<1000 不允许）,3.00,3.400000e+00,1,False


🤔 **先想一想**：“四舍五入”这条规则是怎么判断预测值“是不是四舍五入出来的”？它看的是**预测值的小数位数**（`_decimals`）。这个判断方法有什么隐含假设？

<details><summary>分析</summary>

假设是：如果预测值只有 1～4 位小数，那它“可能是”对标准值四舍五入得到的；于是把标准值按同样的位数取整，相等就算对。

这个推断是单向的、乐观的：`0.3` 也可能是一个算错的比率恰好等于 0.3。规则实际上是在说“**只要预测值等于标准值的某种精度的近似，就算对**”。精度越低（1 位小数），放过错误答案的概率越大。`check_dataset` 只检查了“错误写法保留 2 位小数后仍可区分”，没有检查 1 位小数，也没有检查大金额取整。

这不是说规则错了：用户问“转化率是多少”，模型回答 `31.4%` 本来就该算对。这是一个**有意识的取舍**，重要的是知道取舍在哪里。

</details>

### 5.2 第二层：列怎么对应

忽略列名和列顺序意味着：标准答案的每一列，都要在预测结果里**找到**一列与之对应。`column_views` 还会给每个预测列生成几种“视图”（原值、换算成月份、换算成日期键），任何一种视图能对上都行。

In [13]:
show("evals/compare.py", "column_views")

`evals/compare.py` · `column_views`
```python
def column_views(values: list) -> dict[str, list]:
    """
    一列预测值可以参与比较的几种形式：原始值，以及满足条件时换算后的月份、日期

    月份换算要求整列都是年月标签且只有一个年份，否则“1 月”指哪年说不清，不做换算
    """
    views = {"raw": values}
    non_null = [v for v in values if v is not None]
    if not non_null:
        return views
    parsed = [_year_month(v) for v in non_null]
    if all(parsed) and len({p[0] for p in parsed}) == 1:
        views["month"] = [
            None if v is None else float(_year_month(v)[1]) for v in values
        ]
    converted = [_as_date_key(v) for v in non_null]
    if all(c is not None for c in converted):
        views["date"] = [None if v is None else _as_date_key(v) for v in values]
    return views
```

In [14]:
from evals.compare import column_views

display(column_views(["2025-01", "2025-02", None]))
display(column_views([20251225.0, 20251226.0]))
display(column_views(["2024-12", "2025-01"]))  # 跨年：不做月份换算

{'raw': ['2025-01', '2025-02', None], 'month': [1.0, 2.0, None]}

{'raw': [20251225.0, 20251226.0], 'date': ['2025-12-25', '2025-12-26']}

{'raw': ['2024-12', '2025-01']}

### 5.3 第三层：两阶段搜索

列的对应关系不能只看列内容：两列各自都能对上，但组合起来行可能是错位的（华东的销售额配上了华南的名字）。`results_equal` 分两个阶段解决：

In [15]:
show("evals/compare.py", "results_equal")

`evals/compare.py` · `results_equal`
```python
def results_equal(
    pred: list[dict], gold: list[dict], order_matters: bool = False
) -> bool:
    """判断预测结果与标准结果是否等价，两者都是“列名 -> 值”的字典列表"""
    if len(pred) != len(gold):
        return False
    if not gold:
        return True

    gold_cols = list(gold[0].keys())
    pred_cols = list(pred[0].keys())
    if len(pred_cols) < len(gold_cols):
        return False

    gold_data = {c: [normalize(r[c]) for r in gold] for c in gold_cols}
    # 每个预测列可能有多种形式（原始值、换算成月份或日期），键为 (列名, 形式)
    pred_data = {
        (c, view): values
        for c in pred_cols
        for view, values in column_views([normalize(r[c]) for r in pred]).items()
    }

    # 先找出每个标准列可能对应的 ((预测列, 形式), 倍数)
    candidates: list[list[tuple[tuple[str, str], int]]] = []
    for gc in gold_cols:
        options = [
            (key, s)
            for key in pred_data
            for s in SCALES
            if _column_matches(pred_data[key], gold_data[gc], s, order_matters)
        ]
        if not options:
            return False
        candidates.append(options)

    # 再在候选里找一组互不重复（同一预测列只能用一次）、且逐行也能对齐的对应关系
    def search(i: int, used: set[str], chosen: list) -> bool:
        if i == len(gold_cols):
            pred_rows = [
                tuple(pred_data[key][r] for key, _ in chosen) for r in range(len(pred))
            ]
            gold_rows = [
                tuple(gold_data[gc][r] for gc in gold_cols) for r in range(len(gold))
            ]
            return _rows_match(
                pred_rows, gold_rows, tuple(s for _, s in chosen), order_matters
            )
        for key, s in candidates[i]:
            column = key[0]
            if column not in used and search(
                i + 1, used | {column}, chosen + [(key, s)]
            ):
                return True
        return False

    return search(0, set(), [])
```

1. **剪枝**：对每个标准列，先找出所有“单看这一列能对上”的（预测列, 视图, 倍数）候选。只要有一个标准列没有候选，立刻判错
2. **回溯搜索**：`search(i, used, chosen)` 依次给第 i 个标准列挑一个候选，同一个预测列只能用一次；挑满之后再**逐行**检查组合是否对齐，不对齐就回溯换一个候选

这是一个经典的“约束满足 + 回溯”结构。先用便宜的单列检查剪掉大部分组合，再对少量组合做昂贵的整行检查。最坏情况是指数级的，但标准答案通常只有 1～3 列，实际上很快。

下面的例子里，两列单独看都能对上，但行是错位的：

In [16]:
from evals.compare import results_equal

gold = [{"大区": "华东", "销售额": 200.0}, {"大区": "华南", "销售额": 100.0}]
swapped = [{"area": "华东", "amount": 100.0}, {"area": "华南", "amount": 200.0}]
reordered = [
    {"amount": 100.0, "area": "华南", "orders": 7},
    {"amount": 200.0, "area": "华东", "orders": 9},
]
print("行错位：", results_equal(swapped, gold))
print("列名不同、列顺序不同、多一列、行顺序不同：", results_equal(reordered, gold))

行错位： False
列名不同、列顺序不同、多一列、行顺序不同： True


### 5.4 动手找判分器的漏洞

读判分器的代码，最好的方式是**试图骗过它**。下面每个例子都先自己预测结果，再运行。

In [17]:
experiments = {
    "A 计数整体放大 100 倍": ([{"n": 500}], [{"n": 5}]),
    "B 金额整体放大 100 倍": ([{"m": 1234567.0}], [{"m": 12345.67}]),
    "C 比率只给 1 位小数": ([{"r": 0.3}], [{"r": 0.3049}]),
    "D 大金额取整（差 0.24 元）": ([{"m": 14577501}], [{"m": 14577500.76}]),
    "E 同样的两行，预测按 1000.0、1000.4 的顺序": (
        [{"m": 1000.0}, {"m": 1000.4}],
        [{"m": 1000.4}, {"m": 1000.0}],
    ),
    "F 同样的两行，预测按 1000.4、1000.0 的顺序": (
        [{"m": 1000.4}, {"m": 1000.0}],
        [{"m": 1000.4}, {"m": 1000.0}],
    ),
}
pd.DataFrame(
    [
        {"实验": k, "预测": p, "标准": g, "判定": results_equal(p, g)}
        for k, (p, g) in experiments.items()
    ]
)

,实验,预测,标准,判定
0,A 计数整体放大 100 倍,[{'n': 500}],[{'n': 5}],True
1,B 金额整体放大 100 倍,[{'m': 1234567.0}],[{'m': 12345.67}],True
2,C 比率只给 1 位小数,[{'r': 0.3}],[{'r': 0.3049}],True
3,D 大金额取整（差 0.24 元）,[{'m': 14577501}],[{'m': 14577500.76}],True
4,E 同样的两行，预测按 1000.0、1000.4 的顺序,"[{'m': 1000.0}, {'m': 1000.4}]","[{'m': 1000.4}, {'m': 1000.0}]",False
5,F 同样的两行，预测按 1000.4、1000.0 的顺序,"[{'m': 1000.4}, {'m': 1000.0}]","[{'m': 1000.4}, {'m': 1000.0}]",True


<details><summary>分析：A、B 是规则过宽，E、F 是一个真正的 bug</summary>

**A、B：×100 规则对所有列生效。** 设计意图是“比率列允许用百分数表示”（0.314 vs 31.4），但代码里 `SCALES = (1, 100)` 对每一列都试，计数、金额也不例外。一个把件数乘了 100 的 SQL 当然很少见，但把“分”当“元”的金额（生成器内部就是按分计算的）并不是不可能。更稳妥的规则是：只有标准列的值都在 [0, 1] 之间（看起来像比率）时才允许 ×100。

**C、D：四舍五入规则的代价**，见 5.1 的分析，属于有意识的取舍。

**E、F：判定结果依赖行的顺序。** 两组数据完全一样，只是预测结果的行顺序不同，判定却不同。原因在 `_column_matches` 和 `_rows_match` 的多重集合比较用的是**贪心匹配**：

```python
for g in gold_col:
    for i, p in enumerate(remaining):
        if values_equal(p, g, scale):   # 找到第一个“相等”的就拿走
            remaining.pop(i)
```

如果“相等”是传递的（精确相等），贪心没有问题。但这里的相等带容差和四舍五入：`1000.0` 被认为“等于” `1000.4`（大金额取整规则），也等于 `1000.0`。贪心把 `1000.0` 先分给了标准值 `1000.4`，剩下的 `1000.4` 却不“等于”标准值 `1000.0`（它有 1 位小数，按 1 位小数比较 1000.0 ≠ 1000.4），于是判错。正确的做法是**二分图最大匹配**（例如匈牙利算法或简单的增广路径），保证只要存在一种一一对应就判对。

实际影响很小（需要同一列里出现“整数”和“非常接近的非整数”），但它违反了一条判分器应有的性质：**判定结果不应依赖于与题意无关的因素（行顺序）**。这类性质叫“不变性”，很适合写成测试：把预测结果的行随机打乱 N 次，判定都应相同（property-based testing）。

</details>

> **工程启示**：给判分器写测试时，除了“这个例子应该判对 / 判错”，还可以写**性质测试**：自反（结果和自己比相等）、行顺序不变（`order_matters=False` 时打乱行不影响）、列顺序不变、列名不变。`check_dataset` 里的“判定器认为标准结果与自身不同”就是一条自反性检查。

## 6. `scoring.py`：归因、汇总、基线门禁

### 6.1 `CaseResult`：一次作答的完整记录

In [18]:
show("evals/scoring.py", "CaseResult")

`evals/scoring.py` · `CaseResult`
```python
class CaseResult:
    """一道题（一种问法）的评测结果"""

    case_id: str
    category: str
    difficulty: str
    must_pass: bool
    question: str
    variant: str  # question 或 paraphrase_1 ...
    passed: bool = False
    stage: str = STAGE_WRONG_RESULT
    sql: str | None = None
    error: str | None = None
    # 各环节命中情况，None 表示该答题方式不经过此环节（如直接用标准 SQL 作答）
    recall_missing: list[str] | None = None
    filter_missing: list[str] | None = None
    lint_issues: list[str] = field(default_factory=list)
    correct_attempts: int = 0
    llm_calls: int = 0
    tokens: int = 0
    seconds: float = 0.0

    def to_dict(self) -> dict:
        return asdict(self)
```

几个设计细节：

- 用 `@dataclass` 而不是 dict：字段有名字、有类型、有默认值，拼错字段名会立刻报错；`asdict` 一行就能序列化成 JSON
- `recall_missing: list[str] | None`：**`None` 和 `[]` 含义不同**。`[]` 表示“检查过了，什么都没漏”，`None` 表示“这种作答方式不经过召回环节”（gold / trap 自检）。用 `None` 表达“不适用”，避免把“没检查”误读成“没问题”
- 一条记录里同时有结果（`passed`）、过程（`sql`、`lint_issues`、`correct_attempts`）、成本（`llm_calls`、`tokens`、`seconds`）。**评测的价值很大一部分在明细里**，汇总数字只是入口

### 6.2 归因：第一个出错的环节

In [19]:
show("evals/scoring.py", "attribute_failure")

`evals/scoring.py` · `attribute_failure`
```python
def attribute_failure(
    *,
    error: str | None,
    sql: str | None,
    recall_missing: list[str] | None,
    filter_missing: list[str] | None,
    sql_executable: bool,
) -> str:
    """答错时按流水线顺序找出第一个出问题的环节"""
    if error and not sql:
        return STAGE_ERROR
    if recall_missing:
        return STAGE_RECALL
    if filter_missing:
        return STAGE_FILTER
    if not sql or not sql_executable:
        return STAGE_INVALID_SQL
    return STAGE_WRONG_RESULT
```

按流水线顺序检查，命中第一个就停。这是一个简单但有效的启发式：如果召回阶段就没找到 `dim_date`，后面 SQL 写错就是“果”不是“因”。

🤔 **先想一想**：这个归因可靠吗？什么情况下它会给出误导性的结论？

<details><summary>分析</summary>

它依赖 `expect` 的标注是“必要条件”。但 `expect` 标注的是**一种**写法需要的表，而不是**所有**正确写法都需要的表。

真实结果里就有这样的例子（第 10.3 节会看到）：`order_vs_pay_02`“每个月的支付订单数”标注了 `dim_date`，但模型用 `DATE_FORMAT(pay_time, '%Y-%m')` 根本不需要它，结果也对。如果这道题答错了，会被归为“召回遗漏 dim_date”，可真正的原因可能完全不同。

更一般地说：**归因给出的是“相关”，不是“因果”**。“过滤节点删掉了 fact_order”和“模型没加支付过滤”同时发生，但不能证明如果保留了 fact_order 模型就会加上支付过滤。要证明因果，需要做干预实验：手工把 fact_order 放回上下文，重跑生成，看结果是否变对。

</details>

### 6.3 基线门禁

In [20]:
show("evals/scoring.py", "compare_with_baseline")

`evals/scoring.py` · `compare_with_baseline`
```python
def compare_with_baseline(
    current: dict,
    baseline: dict,
    max_accuracy_drop: float = 0.02,
    max_cost_increase: float = 0.3,
) -> RegressionReport:
    """
    判断本次评测相对基线是否退化：
    - 任一必过题失败
    - 整体准确率下降超过 max_accuracy_drop
    - 平均 LLM 调用次数上升超过 max_cost_increase（比例）
    """
    reasons = []
    if current["must_pass_failed"]:
        reasons.append(f"必过题失败：{current['must_pass_failed']}")
    drop = baseline["accuracy"] - current["accuracy"]
    if drop > max_accuracy_drop:
        reasons.append(
            f"准确率下降 {drop:.1%}（{baseline['accuracy']:.1%} → {current['accuracy']:.1%}）"
        )
    base_calls = baseline.get("avg_llm_calls") or 0
    if base_calls and current["avg_llm_calls"] > base_calls * (1 + max_cost_increase):
        reasons.append(
            f"平均 LLM 调用次数上升（{base_calls} → {current['avg_llm_calls']}）"
        )

    base_cases, cur_cases = baseline.get("per_case", {}), current["per_case"]
    common = set(base_cases) & set(cur_cases)
    newly_failed = sorted(k for k in common if base_cases[k] and not cur_cases[k])
    newly_passed = sorted(k for k in common if not base_cases[k] and cur_cases[k])
    return RegressionReport(bool(reasons), reasons, newly_failed, newly_passed)
```

三个退化条件：必过题失败、整体准确率下降超过阈值（默认 2 个百分点）、平均 LLM 调用次数上升超过 30%。`newly_failed` / `newly_passed` 只在两次都出现的题上比较。

这里有一个值得动手验证的问题。仓库里的基线 `evals/baselines/glm-5.3-flash.json` 是**带换问法**的 100 次作答（91%）。如果你只跑原问法（50 次），或者用 `--ids` 只跑几道题，再和这个基线比较，会发生什么？

In [21]:
from evals.scoring import CaseResult, compare_with_baseline, summarize

baseline = json.loads(
    (ROOT / "evals/baselines/glm-5.3-flash.json").read_text(encoding="utf-8")
)
print("基线：", baseline["total"], "次作答，准确率", baseline["accuracy"])

# 用基线自己的明细构造一次“只跑原问法”的评测，并故意让 3 道原本答对的题答错
rows = [
    json.loads(line)
    for line in (ROOT / "evals/results/glm-5.3-flash/baseline/results.jsonl")
    .read_text(encoding="utf-8")
    .splitlines()
]
only_question = [CaseResult(**r) for r in rows if r["variant"] == "question"]
broken = {"order_vs_pay_02", "date_role_02", "region_role_02"}  # 都不是必过题
for r in only_question:
    if r.case_id in broken:
        r.passed = False
current = summarize(only_question)
report = compare_with_baseline(current, baseline)
print("本次：", current["total"], "次作答，准确率", current["accuracy"])
print("判定退化：", report.regressed, report.reasons)
print("由对变错：", report.newly_failed)

基线： 100 次作答，准确率 0.91
本次： 50 次作答，准确率 0.9
判定退化： True ["必过题失败：['activity_01']"]
由对变错： ['date_role_02#question', 'order_vs_pay_02#question', 'region_role_02#question']


<details><summary>分析</summary>

门禁确实判了“退化”，但**理由是错的**：它说必过题 `activity_01` 失败了，而 `activity_01` 在基线里本来就是失败的（基线的 `must_pass_failed` 就是 `['activity_01']`）。真正由对变错的 3 道题，反而没有成为退化的理由：本次 45/50 = 90%，基线 91%，只下降 1 个百分点，没有超过 2 个百分点的阈值。

这里叠加了两个问题，下一个单元格分别验证。

</details>
</details>

In [22]:
# 问题一：拿基线和它自己比
self_check = compare_with_baseline(baseline, baseline)
print("基线 vs 基线：退化 =", self_check.regressed, self_check.reasons)

# 问题二：去掉必过题的干扰，只看准确率条件
no_must = {**current, "must_pass_failed": []}
print("只看准确率：退化 =", compare_with_baseline(no_must, baseline).regressed)

基线 vs 基线：退化 = True ["必过题失败：['activity_01']"]
只看准确率：退化 = False


<details><summary>分析</summary>

**问题一：门禁不满足自反性。** 拿基线和它自己比，结论是“退化”。原因是“必过题失败”这一条检查的是**本次**有没有必过题失败，而不是“有没有**新增**的必过题失败”。只要基线里有一道必过题没过（现在就是这样），以后每一次评测都会被判退化、退出码都是 1。一个永远报警的门禁和没有门禁效果一样：大家很快就会习惯性忽略它（“狼来了”）。修复方法：只把 `current.must_pass_failed - baseline.must_pass_failed` 当作退化理由，已知失败的必过题单独列出提醒。

**问题二：比较了两个不同题集的准确率。** 去掉必过题的干扰后，3 道题由对变错却“未退化”。原因是两次评测的**题集不同**：基线包含 50 次换问法作答，换问法的准确率（86%）比原问法（96%）低 10 个点，把基线整体拉低了。拿“原问法”的分数和“原问法 + 换问法”的基线比，天然就有 5 个点的“虚假余量”。修复思路：只在两次都出现的题（`per_case` 的交集）上计算准确率，或者在题集不一致时直接报错，要求用同样的参数重跑基线。

两个问题的共同教训：**比较两个东西之前，先确认它们在度量同一样东西；写一个比较函数时，先测“自己和自己比”**。`tests/test_scoring.py::test_compare_with_baseline` 用的是构造数据，基线里没有失败的必过题，所以没测出问题一。

</details>

### 6.4 多轮汇总

多轮复用了单轮的大部分结构：`TurnResult` 继承 `CaseResult`，多了意图、改写、直接回答、反问；`summarize_dialogs` 产出的字段名（`accuracy`、`per_case`、`avg_llm_calls`、`must_pass_failed`）与单轮相同，所以 `compare_with_baseline` 不用改就能用于多轮。**让新功能遵守旧接口**，是减少代码改动面的常用手法。

In [23]:
show("evals/scoring.py", "attribute_turn_failure")

`evals/scoring.py` · `attribute_turn_failure`
```python
def attribute_turn_failure(
    *,
    intent_correct: bool | None,
    error: str | None,
    sql: str | None,
    recall_missing: list[str] | None,
    filter_missing: list[str] | None,
    sql_executable: bool,
) -> str:
    """多轮的一轮答错时，先看意图是否判断错，再按单轮流水线顺序归因"""
    if error and not sql:
        return STAGE_ERROR
    if intent_correct is False:
        return STAGE_INTENT
    return attribute_failure(
        error=error,
        sql=sql,
        recall_missing=recall_missing,
        filter_missing=filter_missing,
        sql_executable=sql_executable,
    )
```

## 7. `runner.py`：把一切串起来

`runner.py` 是外壳，重点看它怎么处理“编排”问题：并发、观测、开关、输出。

### 7.1 怎么拿到 Agent 内部每个节点的输出

In [24]:
show("evals/runner.py", "run_agent")

`evals/runner.py` · `run_agent`
```python
async def run_agent(question: str, today: str) -> dict:
    """用问数图回答一个问题，返回最终 SQL 以及各节点的输出，供打分和归因使用"""
    outputs: dict[str, dict] = {}
    usage = UsageCallback()
    error = None
    started = time.perf_counter()
    async with (
        meta_mysql_client_manager.session_factory() as meta_session,
        dw_mysql_client_manager.session_factory() as dw_session,
    ):
        context = QueryService.from_sessions(meta_session, dw_session).build_context()
        try:
            async for mode, chunk in graph.astream(
                input={"query": question, "today": today},
                context=context,
                stream_mode=["updates", "custom"],
                config={"callbacks": [usage]},
            ):
                if mode == "updates":
                    for node, update in chunk.items():
                        if isinstance(update, dict):
                            outputs[node] = update
        except Exception as e:
            error = f"{type(e).__name__}: {e}"
    # 校正后的 SQL 是最后一版；校正节点没有执行过时，用生成节点的 SQL
    sql = (outputs.get("correct_sql") or outputs.get("generate_sql") or {}).get("sql")
    return {
        "sql": sql,
        "error": error,
        "outputs": outputs,
        "llm_calls": usage.calls,
        "tokens": usage.tokens,
        "seconds": time.perf_counter() - started,
    }
```

- `graph.astream(stream_mode=["updates", "custom"])`：`updates` 模式会在**每个节点执行完后**给出这个节点返回的 dict。runner 把它们按节点名收集到 `outputs`，之后从 `outputs["merge_retrieved_info"]` 里取召回到的表、从 `outputs["filter_table"]` 里取过滤后的表，用于归因。这就是“**可观测性**”：评测不只看最终输出，还能看到中间每一步
- 注意它**没有改动问数图的任何代码**就拿到了中间结果。LangGraph 的流式接口天然提供了这个能力，评测代码和业务代码保持解耦
- `UsageCallback` 是 LangChain 的回调：每次调用模型前后被触发，用来计数和累加 token。同样是不改业务代码就能“埋点”
- 异常被捕获成 `error` 字符串而不是向外抛：一道题崩了不应该让整个评测中断。这和问数图节点里 `raise` 的策略正好相反——**业务流程要快速失败，批处理要隔离失败**

`sql = (outputs.get("correct_sql") or outputs.get("generate_sql") or {}).get("sql")`：取最后一版 SQL。

🤔 **先想一想**：评测拿到 SQL 后，是用 Agent 自己 `run_sql` 节点的执行结果打分，还是自己重新执行一遍？两种做法分别度量的是什么？

<details><summary>分析</summary>

看 `evaluate_case`：它调用 `execute(result.sql)` 自己重新执行。这样做的好处是统一：gold / trap / agent 三种作答方式走同一条执行路径，判分条件完全一致。

代价是它度量的是“**最后一版 SQL 对不对**”，而不完全是“用户看到的结果对不对”。两者在几种情况下会不同：

- Agent 的 `run_sql` 最多返回 `agent.max_result_rows`（1000）行，评测执行不截断
- Agent 执行超时（`max_execution_ms`）时用户看到的是报错，评测执行没有超时
- SQL 里用了 `CURDATE()` / `NOW()`：Agent 执行时和评测执行时的“当前日期”都是**跑评测那天的真实日期**，而不是题目的 `today`。这一点在第 10.5 节会看到实际影响

这不是错，而是需要写清楚的“**度量的定义**”。

</details>

### 7.2 并发与标准答案缓存

In [25]:
show("evals/runner.py", "run")

`evals/runner.py` · `run`
```python
async def run(args) -> int:
    multiturn = bool(args.multiturn)
    dataset = args.dataset or (MULTITURN_DATASET if multiturn else DEFAULT_DATASET)
    items = select_items(
        load_dataset(dataset),
        set(args.ids.split(",")) if args.ids else None,
        args.category,
        multiturn,
    )
    tasks = []
    if multiturn:
        # 多轮按整段对话调度；对话内部的各轮必须按顺序回答
        tasks = [(item, None, None) for item in items]
    else:
        for item in items:
            turn = item["turns"][0]
            tasks.append((item, "question", turn["question"]))
            if args.paraphrases:
                for i, p in enumerate(turn.get("paraphrases", []), start=1):
                    tasks.append((item, f"paraphrase_{i}", p))

    # 消融开关直接改运行时配置，节点每次执行都会读取
    app_config.knowledge.enable_examples = not args.no_examples
    app_config.agent.semantic_check = not args.no_semantic_check
    cache = None
    if args.llm_cache:
        cache = SQLiteLLMCache(Path(args.llm_cache))
        set_llm_cache(cache)

    dw_mysql_client_manager.init()
    if args.predictor == "agent":
        meta_mysql_client_manager.init()
        qdrant_client_manager.init()
        es_client_manager.init()
        embedding_client_manager.init()

    gold_cache: dict = {}
    semaphore = asyncio.Semaphore(args.concurrency)
    done = 0

    async def worker(item, variant, question):
        nonlocal done
        async with semaphore:
            if multiturn:
                from evals.multiturn import evaluate_dialog

                turns = await evaluate_dialog(
                    item, args.predictor, args.multiturn, gold_cache
                )
                done += 1
                marks = " ".join("✓" if t.passed else f"✗{t.stage}" for t in turns)
                print(f"[{done}/{len(tasks)}] {item['id']} {marks}")
                return turns
            r = await evaluate_case(item, variant, question, args.predictor, gold_cache)
            done += 1
            print(
                f"[{done}/{len(tasks)}] {r.case_id}#{r.variant} {'✓' if r.passed else '✗ ' + r.stage}"
            )
            return [r]

    try:
        # 先串行算出所有标准答案，避免并发时重复执行；多轮题按（题目, 轮次）缓存
        for item in items:
            if multiturn:
                for t, turn in enumerate(item["turns"]):
                    gold_cache[(item["id"], t)] = await execute(turn["gold_sql"])
            else:
                gold_cache[item["id"]] = await execute(item["turns"][0]["gold_sql"])
        results = [
            r for rs in await asyncio.gather(*(worker(*t) for t in tasks)) for r in rs
        ]
    finally:
        await dw_mysql_client_manager.close()
        if args.predictor == "agent":
            await meta_mysql_client_manager.close()
            await qdrant_client_manager.close()
            await es_client_manager.close()

    summary = summarize_dialogs(results) if multiturn else summarize(results)
    run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
    meta = {
        "run_id": run_id,
        "predictor": args.predictor,
        "dataset": str(dataset.resolve().relative_to(ROOT)),
        "examples": app_config.knowledge.enable_examples,
        "semantic_check": app_config.agent.semantic_check,
        "llm_model": app_config.llm.model_name,
    }
    if multiturn:
        meta["mode"] = args.multiturn
    if cache:
        meta["llm_cache"] = {"hits": cache.hits, "misses": cache.misses}

    regression = None
    baseline_path = Path(args.baseline) if args.baseline else None
    if baseline_path and baseline_path.exists() and not args.save_baseline:
        regression = compare_with_baseline(
            summary,
            json.loads(baseline_path.read_text(encoding="utf-8")),
            args.max_accuracy_drop,
        )

    out_dir = Path(args.out) / run_id
    out_dir.mkdir(parents=True, exist_ok=True)
    with (out_dir / "results.jsonl").open("w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r.to_dict(), ensure_ascii=False, default=str) + "\n")
    (out_dir / "summary.json").write_text(
        json.dumps({"meta": meta, **summary}, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    render = render_dialog_markdown if multiturn else render_markdown
    report = render(summary, results, meta, regression)
    (out_dir / "summary.md").write_text(report, encoding="utf-8")
    if args.save_baseline and baseline_path:
        baseline_path.write_text(
            json.dumps({"meta": meta, **summary}, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        print(f"已保存基线：{baseline_path}")

    print("\n" + report)
    print(f"报告目录：{out_dir}")
    return 1 if regression and regression.regressed else 0
```

值得学习的几处：

1. **先串行算标准答案，再并发答题**：标准答案被多次使用（原问法和换问法共用一份），如果在并发任务里懒加载，几个任务可能同时发现“缓存里没有”然后重复执行。先串行填好缓存，后面只读，就彻底避开了这个竞态
2. **`asyncio.Semaphore(args.concurrency)` 限流**：`gather` 会一次性启动所有协程，信号量保证同时最多 4 个在跑。LLM 平台有速率限制，TEI 在 CPU 上也很慢，并发太高反而更慢
3. **多轮按整段对话调度**：对话内部的轮次有先后依赖，只能顺序执行；不同对话之间可以并发。**并发的粒度要和依赖关系对齐**
4. **消融开关直接改运行时配置**：`app_config.knowledge.enable_examples = not args.no_examples`。简单直接，但它修改的是模块级单例，同一个进程里没法同时跑“开”和“关”两组。这是为了简单而接受的限制
5. **`try / finally` 关闭客户端**：无论成功失败都释放连接
6. **退出码**：退化时 `return 1`。这让 runner 可以直接放进 CI：CI 看到非 0 退出码就会把这次提交标成失败
7. **报告三件套**：`results.jsonl`（每行一个 JSON，便于流式追加和用 pandas 读取）、`summary.json`（机器读）、`summary.md`（人读）。meta 里记录了模型名、开关、缓存命中数——**任何结果都要能回答“它是在什么条件下跑出来的”**

## 8. `llm_cache.py`：录制回放与可复现性

In [26]:
show("evals/llm_cache.py", "SQLiteLLMCache")

`evals/llm_cache.py` · `SQLiteLLMCache`
```python
class SQLiteLLMCache(BaseCache):
    """以 SQLite 文件持久化的 LLM 缓存"""

    def __init__(self, path: Path):
        path.parent.mkdir(parents=True, exist_ok=True)
        self._lock = threading.Lock()
        self._conn = sqlite3.connect(path, check_same_thread=False)
        self._conn.execute(
            "create table if not exists llm_cache (key text primary key, value text not null)"
        )
        self._conn.commit()
        self.hits = 0
        self.misses = 0

    @staticmethod
    def _key(prompt: str, llm_string: str) -> str:
        return hashlib.sha256(f"{llm_string}\n{prompt}".encode()).hexdigest()

    def lookup(self, prompt: str, llm_string: str) -> RETURN_VAL_TYPE | None:
        with self._lock:
            row = self._conn.execute(
                "select value from llm_cache where key = ?",
                (self._key(prompt, llm_string),),
            ).fetchone()
        if row is None:
            self.misses += 1
            return None
        self.hits += 1
        return loads(row[0])

    def update(self, prompt: str, llm_string: str, return_val: RETURN_VAL_TYPE) -> None:
        with self._lock:
            self._conn.execute(
                "insert or replace into llm_cache (key, value) values (?, ?)",
                (self._key(prompt, llm_string), dumps(return_val)),
            )
            self._conn.commit()

    def clear(self, **kwargs: Any) -> None:
        with self._lock:
            self._conn.execute("delete from llm_cache")
            self._conn.commit()

    # 读写都很快，直接在事件循环里同步执行，避免线程池里并发访问同一个连接
    async def alookup(self, prompt: str, llm_string: str) -> RETURN_VAL_TYPE | None:
        return self.lookup(prompt, llm_string)

    async def aupdate(
        self, prompt: str, llm_string: str, return_val: RETURN_VAL_TYPE
    ) -> None:
        self.update(prompt, llm_string, return_val)
```

它实现的是 LangChain 的 `BaseCache` 接口：调用模型前先 `lookup(prompt, llm_string)`，命中就直接返回录好的回答，不命中就真的调用并 `update` 写入。缓存键是 `sha256(模型参数 + 完整提示词)`。

这个模式在测试领域叫 **录制回放（record / replay）**，Ruby 的 VCR、Python 的 vcrpy 都是同一思路。它带来三个好处：

1. **可复现**：同样的提示词一定得到同样的回答，重跑评测结果不变
2. **便宜、快**：第一次跑 100 题每题约 105 秒，回放时 17 秒
3. **只改判分逻辑时可以“重新打分”**：改了 `compare.py` 之后，用缓存回放就能看到新规则对历史作答的影响，而不需要重新调用模型

但它有一个苛刻的前提：**提示词必须逐字节确定**。哪怕一个空格、一个列表顺序不同，键就变了，缓存失效。项目里踩过的坑：

- `extract_keywords` 和召回节点原来用 `set` 去重，Python 的字符串哈希每个进程随机化（`PYTHONHASHSEED`），`set` 的迭代顺序每次启动都可能不同 → 关键词顺序不同 → 提示词不同 → 换进程后缓存几乎全部失效。改成 `dict.fromkeys` 保序去重后解决
- TEI 把并发请求合批计算，批次不同时同一段文本的向量有约 1e-7 的差异 → 召回结果里分数几乎相同的两项可能换了顺序 → 提示词变了。这就是“2.6% 不命中”的来源，目前还没解决

In [27]:
import sqlite3

cache_path = ROOT / "evals/.cache/glm-5.3-flash.sqlite"
if cache_path.exists():
    conn = sqlite3.connect(cache_path)
    n = conn.execute("select count(*) from llm_cache").fetchone()[0]
    print(f"缓存文件 {cache_path.stat().st_size / 1e6:.1f} MB，{n} 条录制的 LLM 回答")
else:
    print("没有找到缓存文件 evals/.cache/glm-5.3-flash.sqlite")

缓存文件 2.5 MB，1168 条录制的 LLM 回答


🤔 **先想一想**：每次评测的 `summary.json` 都记录了 `llm_cache.hits / misses`。下表是 7 次已提交评测的命中情况，你能从中读出什么？

In [28]:
RESULTS = ROOT / "evals" / "results" / "glm-5.3-flash"
RUNS = [
    "baseline",
    "no_examples",
    "no_lint",
    "mt_teacher_phase1",
    "mt_free_phase1",
    "mt_teacher",
    "mt_free",
]


def load_run(name):
    folder = RESULTS / name
    s = json.loads((folder / "summary.json").read_text(encoding="utf-8"))
    r = pd.DataFrame(
        [
            json.loads(x)
            for x in (folder / "results.jsonl").read_text(encoding="utf-8").splitlines()
        ]
    )
    return s, r


runs = {name: load_run(name) for name in RUNS}
pd.DataFrame(
    [
        {
            "评测": name,
            "运行时间": s["meta"]["run_id"],
            "作答次数": s["total"],
            "准确率": s["accuracy"],
            "缓存命中": s["meta"]["llm_cache"]["hits"],
            "未命中": s["meta"]["llm_cache"]["misses"],
            "未命中率": round(
                s["meta"]["llm_cache"]["misses"]
                / (s["meta"]["llm_cache"]["hits"] + s["meta"]["llm_cache"]["misses"]),
                3,
            ),
        }
        for name, (s, _) in runs.items()
    ]
)

,评测,运行时间,作答次数,准确率,缓存命中,未命中,未命中率
0,baseline,20261126-141202,100,0.9100,589,16,0.026
1,no_examples,20261126-141359,50,0.9600,304,2,0.007
2,no_lint,20261126-141653,50,0.9600,299,1,0.003
3,mt_teacher_phase1,20261126-141904,44,0.9318,284,3,0.010
4,mt_free_phase1,20261126-142046,44,0.9091,286,1,0.003
5,mt_teacher,20261126-143225,48,0.9375,210,94,0.309
6,mt_free,20261126-143627,48,0.9583,243,66,0.214


<details><summary>分析</summary>

- 单轮三组和多轮第一阶段的未命中率都很低（0.3%～2.6%），说明它们是**回放**：模型的回答绝大部分来自之前录制的那一次运行
- `mt_teacher` / `mt_free` 未命中 31% / 21%：多轮第 4、5 步改了改写提示词（新增 clarify 意图），所有追问轮的改写调用、以及因为参考 SQL 而变化的生成调用都是第一次调用。这两组是**真实调用为主**的结果

这有一个推论，第 10.4 节会用到：**回放出来的失败不是独立样本**。同一条录制的错误回答，在 5 次回放评测里会“失败” 5 次，但它只发生过一次。

</details>

## 9. `multiturn.py`：多轮评测

多轮评测要解决两个单轮没有的问题：**历史从哪来**、**会话中途暂停（反问）怎么办**。

In [29]:
show("evals/multiturn.py", "run_session_turn")

`evals/multiturn.py` · `run_session_turn`
```python
async def run_session_turn(
    thread_id: str,
    question: str,
    today: str,
    seed_history: list[Turn] | None = None,
    seed_last_result: list[dict] | None = None,
    clarify_reply: str | None = None,
) -> dict:
    """
    在会话里回答一轮，返回意图、改写、最终 SQL 或直接回答，以及单轮流水线各节点的输出

    - seed_history / seed_last_result 只在 teacher 模式下使用：作为新会话的初始历史和上一轮结果
    - 会话图停在反问处时，扮演用户用 clarify_reply 回答（没有标注时用默认回答）并恢复执行，
      每轮最多恢复一次；意图按第一次判断计（是否该反问），改写结果按最后一次计
    """
    usage = UsageCallback()
    pipeline_outputs: dict[str, dict] = {}
    session_outputs: dict[str, dict] = {}
    intents: list[str] = []
    clarify_question = None
    error = None
    started = time.perf_counter()
    graph_input = {
        "user_input": question,
        "today": today,
        "clarified": False,
        "clarification": None,
    }
    if seed_history:
        graph_input["history"] = seed_history
    if seed_last_result is not None:
        graph_input["last_result"] = seed_last_result
    config = {"configurable": {"thread_id": thread_id}, "callbacks": [usage]}
    async with (
        meta_mysql_client_manager.session_factory() as meta_session,
        dw_mysql_client_manager.session_factory() as dw_session,
    ):
        context = QueryService.from_sessions(meta_session, dw_session).build_context()
        try:
            for _ in range(2):
                interrupted = None
                async for namespace, mode, chunk in session_graph.astream(
                    input=graph_input,
                    config=config,
                    context=context,
                    stream_mode=["updates", "custom"],
                    subgraphs=True,
                ):
                    if mode != "updates":
                        continue
                    if not namespace and "__interrupt__" in chunk:
                        interrupted = chunk["__interrupt__"][0].value
                        continue
                    # namespace 为空是会话图自己的节点，否则是 run_pipeline 里单轮图的节点
                    target = pipeline_outputs if namespace else session_outputs
                    for node, update in chunk.items():
                        if isinstance(update, dict):
                            target[node] = update
                            if node == "contextualize" and not namespace:
                                intents.append(update.get("intent"))
                if interrupted is None:
                    break
                clarify_question = interrupted.get("question")
                graph_input = Command(resume=clarify_reply or DEFAULT_CLARIFY_REPLY)
        except Exception as e:
            error = f"{type(e).__name__}: {e}"

    rewrite = session_outputs.get("contextualize") or {}
    pipeline = session_outputs.get("run_pipeline") or {}
    answer = (session_outputs.get("answer_from_result") or {}).get("answer")
    # 修正次数用完时 run_pipeline 不返回 SQL，此时用最后一版候选 SQL 判断是否可执行
    sql = pipeline.get("sql") or (
        pipeline_outputs.get("correct_sql")
        or pipeline_outputs.get("generate_sql")
        or {}
    ).get("sql")
    return {
        "intent": intents[0] if intents else None,
        "standalone": rewrite.get("standalone"),
        "clarify_question": clarify_question,
        "answer": answer if not sql else None,
        "sql": sql,
        "error": error or pipeline.get("error"),
        "outputs": pipeline_outputs,
        "llm_calls": usage.calls,
        "tokens": usage.tokens,
        "seconds": time.perf_counter() - started,
    }
```

### 9.1 两种模式：teacher forcing vs 自由对话

这两个名字来自序列模型训练：teacher forcing 指训练时每一步都喂给模型“标准的上一步”，而不是模型自己生成的上一步。

| | teacher | free |
| --- | --- | --- |
| 会话 | 每一轮都新开一个 `thread_id` | 整段对话用同一个 `thread_id` |
| 历史 | `gold_turn()` 用标准改写、标准 SQL 及其执行结果构造 `seed_history` | 模型自己前几轮的输出 |
| 上一轮结果 | `seed_last_result` = 上一轮标准 SQL 的结果 | 会话图自己保存的 `last_result` |
| 度量 | 单轮追问能力（每一轮都是公平的考题） | 真实体验（错误会传递） |

`seed_history` 能直接作为图的输入，是因为 `SessionState.history` 有 `operator.add` reducer：新会话的初始状态为空，输入里的 `history` 被“追加”进去，正好就是预填的历史。**理解了状态的 reducer 语义，才能用这种方式注入测试数据。**

### 9.2 反问的处理：最多恢复一次

```python
for _ in range(2):
    async for namespace, mode, chunk in session_graph.astream(..., subgraphs=True):
        if not namespace and "__interrupt__" in chunk:
            interrupted = chunk["__interrupt__"][0].value
    if interrupted is None:
        break
    graph_input = Command(resume=clarify_reply or DEFAULT_CLARIFY_REPLY)
```

- `subgraphs=True` 让流里同时带上子图（单轮问数图）的事件，`namespace` 为空表示会话图自己的节点，非空表示 `run_pipeline` 里的单轮图节点。这样一次流式调用就同时拿到了会话层（意图、改写）和流水线层（召回、过滤、SQL）的输出
- 会话图停在 `interrupt` 时，评测扮演用户：用题目标注的 `clarify_reply`（该问的轮次），或者默认回答“按你的理解直接查询”（不该问却问了的轮次）。`for _ in range(2)` 保证最多恢复一次，**任何循环都要有上界**，防止图的 bug 让评测卡死
- 意图记录的是**第一次**判断（`intents[0]`），用来衡量“该不该反问”；改写记录的是最后一次（反问之后的）

### 9.3 直接回答怎么判分

In [30]:
show("evals/multiturn.py", "answer_failure_stage")
src = (ROOT / "evals/multiturn.py").read_text(encoding="utf-8")
start = src.index("        executable = False\n        if result.answer")
print(src[start : src.index("        result.stage =", start)])

`evals/multiturn.py` · `answer_failure_stage`
```python
def answer_failure_stage(result: TurnResult) -> str:
    """直接回答答错时的归因：意图判错优先，否则就是回答的内容不对"""
    return STAGE_INTENT if result.intent_correct is False else STAGE_WRONG_RESULT
```

        executable = False
        if result.answer and not result.sql:
            # 直接基于上一轮结果用文字回答：回答里要出现标注的关键内容
            contains = turn.get("gold_answer_contains") or []
            result.passed = bool(contains) and all(c in result.answer for c in contains)
        elif result.sql:
            try:
                pred_rows = await execute(result.sql)
                executable = True
                result.passed = results_equal(
                    pred_rows, gold_rows, bool(turn.get("order_matters"))
                )
            except Exception as e:
                result.error = (
                    result.error or f"执行失败：{getattr(e, 'orig', None) or e}"
                )



直接回答（没有 SQL）时，判分标准是“回答里出现了 `gold_answer_contains` 的每一项”。

🤔 **先想一想**：`mt_result_ref_01` 第 2 轮问“排第一的是哪个大区”，`gold_answer_contains: [华东]`。下面哪些回答会被判对？

1. “排第一的是华东。”
2. “华东排第二，排第一的是华南。”
3. “华东最高，约占全部的 41%。”（真实占比是 31.40%）

<details><summary>分析</summary>

三个都会被判对。子串匹配只能确认“提到了正确的实体”，不能确认“说对了关于它的事”，更不检查回答里的数字。真实结果里模型的回答是“华东，14577500.76 元，约占 31.40%”，数字恰好都对，但评测并没有验证这一点。

改进方向（按成本从低到高）：
- 标注时同时给出必须出现的数字（`31.4`），并允许一定的格式差异
- 让模型输出结构化回答（`{"entity": "华东", "value": 14577500.76}`），用 `compare.py` 判分
- 用另一个 LLM 做裁判（LLM-as-judge），但它本身也需要被评测

这是“**用文字回答的系统难以自动评测**”的普遍困境。目前只有 2 轮是直接回答，影响很小，但随着这类轮次增多，判分方式需要升级。

</details>

## 10. 带着问题重读真实结果

`eval_tutorial.ipynb` 第 9、10 节已经给出了结果和解读。这一节不重复结论，而是**演示得出（或质疑）结论的过程**。每个小节都从一个具体的疑问开始。

### 10.1 第一个问题永远是：这个数字是怎么来的？

In [31]:
base_s, base_r = runs["baseline"]
print("meta:", json.dumps(base_s["meta"], ensure_ascii=False))
base_r.groupby("variant")["passed"].agg(["count", "sum", "mean"])

meta: {"run_id": "20261126-141202", "predictor": "agent", "dataset": "evals/datasets/traps.yaml", "examples": true, "semantic_check": true, "llm_model": "glm-5.3-flash", "llm_cache": {"hits": 589, "misses": 16}}


,count,sum,mean
variant,,,
paraphrase_1,50,43,0.86
question,50,48,0.96


“单轮 91%”是 100 次作答的平均，但这 100 次由两种性质不同的作答组成：原问法 96%（48/50），换问法 86%（43/50）。**一个平均数掩盖了两个群体的差异**，报告准确率时应该把两个都写出来，而不只是 91%。

### 10.2 这个差异是真的吗？

原问法和换问法差了 10 个百分点。在下结论“模型对措辞敏感”之前，先问：50 道题的样本，10 个点的差异可能只是运气吗？

先看单个准确率的不确定性（Wilson 置信区间，比教科书上的正态近似在 p 接近 0 或 1 时更准确）：

In [32]:
def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    # 二项比例的 Wilson 95% 置信区间
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - half, center + half


for label, k, n in [
    ("原问法", 48, 50),
    ("换问法", 43, 50),
    ("全部", 91, 100),
    ("多轮追问轮（同一批 20 段）", 24, 24),
]:
    lo, hi = wilson(k, n)
    print(f"{label:<16} {k}/{n} = {k / n:.0%}   95% 置信区间 [{lo:.0%}, {hi:.0%}]")

原问法              48/50 = 96%   95% 置信区间 [87%, 99%]
换问法              43/50 = 86%   95% 置信区间 [74%, 93%]
全部               91/100 = 91%   95% 置信区间 [84%, 95%]
多轮追问轮（同一批 20 段）  24/24 = 100%   95% 置信区间 [86%, 100%]


区间很宽：原问法的真实准确率可能在 87%～99% 之间。两个区间有重叠，但**这两组不是独立的**：它们是同一批题的两种问法。对配对数据，正确的做法是只看“结论不一致”的题（McNemar 检验的思路）：

In [33]:
paired = base_r.pivot_table(
    index="case_id", columns="variant", values="passed", aggfunc="first"
)
only_q = paired[(paired["question"]) & (~paired["paraphrase_1"])].index.tolist()
only_p = paired[(~paired["question"]) & (paired["paraphrase_1"])].index.tolist()
print("原问法对、换问法错：", only_q)
print("原问法错、换问法对：", only_p)

# 符号检验（McNemar 精确版）：在不一致的题里，如果两种问法一样难，每道题偏向哪边是 50/50
b, c = len(only_q), len(only_p)
n = b + c
p_value = min(1.0, 2 * sum(math.comb(n, i) for i in range(0, min(b, c) + 1)) / 2**n)
print(f"不一致的题 {n} 道，{b} : {c}，双侧 p = {p_value:.3f}")

原问法对、换问法错： ['funnel_03', 'relative_time_02', 'relative_time_04', 'relative_time_05', 'snowflake_03']
原问法错、换问法对： []
不一致的题 5 道，5 : 0，双侧 p = 0.062


<details><summary>怎么理解这个结果</summary>

5 道题全部是“原问法对、换问法错”，没有一道反过来。如果两种问法真的一样难，5 次全部偏向同一边的概率是 2 × (1/2)^5 ≈ 6%。

- 按惯用的 5% 门槛，**还不能说“统计显著”**；但方向非常一致，值得认真对待
- 更重要的是看这 5 道题**为什么**错：如果原因是同一类（比如都和措辞里的“今年”“上月”有关），那就是一个真实的、可以修的问题，而不只是随机波动

这就是定量和定性结合：统计告诉你“这可能不是噪声”，逐题分析告诉你“这是什么”。下一节就做逐题分析。

</details>

### 10.3 逐题看失败：先看 SQL，再看归因

不要从归因表开始，**从 SQL 开始**。下面把每次失败的问题、模型 SQL、错误写法说明放在一起：

In [34]:
trap_by_id = {i["id"]: i["turns"][0] for i in traps}
fails = base_r[~base_r["passed"]].sort_values(["case_id", "variant"])
for _, r in fails.iterrows():
    turn = trap_by_id[r["case_id"]]
    print(f"■ {r['case_id']} · {r['variant']} · 归因：{r['stage']}")
    print(f"  问题：{r['question']}")
    print(f"  这类题的陷阱：{turn['trap']}")
    sql = (
        r["sql"] if isinstance(r["sql"], str) else f"（没有 SQL）{str(r['error'])[:90]}"
    )
    print("  模型 SQL：" + " ".join(sql.split())[:260])
    print()

■ activity_01 · paraphrase_1 · 归因：过滤误删
  问题：参加今年双 11 活动的商品销量
  这类题的陷阱：用活动期间代替活动归因，把期间内未参加活动的商品也算了进去
  模型 SQL：SELECT d.sku_name AS 商品名称, SUM(f.quantity) AS 销量 FROM fact_order_item f JOIN dim_activity a ON f.activity_id = a.activity_id JOIN dim_sku d ON f.sku_id = d.sku_id WHERE a.activity_name = '2025年双11' GROUP BY d.sku_id, d.sku_name ORDER BY 销量 DESC

■ activity_01 · question · 归因：过滤误删
  问题：2025 年双 11 活动的商品一共卖了多少件？
  这类题的陷阱：用活动期间代替活动归因，把期间内未参加活动的商品也算了进去
  模型 SQL：SELECT SUM(foi.quantity) AS 销量 FROM fact_order_item foi INNER JOIN dim_activity da ON foi.activity_id = da.activity_id WHERE da.activity_name = '2025年双11'

■ fanout_04 · paraphrase_1 · 归因：结果不符
  问题：今年每个品牌的销量是多少件
  这类题的陷阱：用订单级的总件数按明细关联后求和，件数被重复计算且被算到所有品牌上
  模型 SQL：SELECT s.brand_name AS 品牌, SUM(foi.quantity) AS 销量 FROM fact_order_item foi JOIN fact_order fo ON foi.order_id = fo.order_id JOIN dim_sku s ON foi.sku_id = s.sku_id WHERE fo.order_time >= '2025-01-01' AND fo.order_time < '2026-01-01' GROUP BY s.brand_name ORDE

■ fano

🤔 **先想一想**：在看下面的分析之前，试着把这 9 次失败按“真正的原因”分组。提示：看 WHERE 条件里用的是哪个日期字段、有没有 `pay_time IS NOT NULL`、有没有 `CURDATE()`。

<details><summary>我的分组</summary>

| 真正的原因 | 失败 | 归因给出的环节 |
| --- | --- | --- |
| “销量 / 卖了多少件”没有限定已支付，按下单日期统计 | `fanout_04` ×2、`snowflake_03` 换问法、`activity_01` ×2 | 结果不符 / 过滤误删 |
| SQL 里用了 `CURDATE()` / `CURRENT_DATE`，没用注入的“今天” | `relative_time_02`、`_04` 换问法 | 结果不符 |
| 模型输出的 JSON 里有全角引号，流程中断 | `relative_time_05` 换问法 | 流程异常 |
| 指标定义前后矛盾（知识库的问题，不是模型的） | `funnel_03` 换问法 | 结果不符 |

几个观察：

1. **9 次失败里 5 次是同一个原因**（“销量”的口径），而归因表把它们分到了“结果不符”和“过滤误删”两类。按归因环节统计会让你以为问题很分散，按根因统计才发现问题很集中。这就是为什么说“归因只是第一步，还是要回到 SQL”
2. **`activity_01` 换问法“参加今年双 11 活动的商品销量”**：模型按商品分组返回了每个商品的销量，标准答案是一个总数。这个问法本身就有两种合理的理解，**这是题目的问题**。换问法也需要像标准问法一样审核“是否只有一种理解”
3. 换问法失败的 5 题里，2 题和相对时间有关（“上月”“过去一周”），1 题是 JSON。也就是说 10.2 节的“措辞敏感”，有相当一部分其实是“措辞里的相对时间触发了 `CURDATE()`”这一个具体问题

</details>

### 10.4 “剩余失败约一半是 JSON 解析” —— 这句话准确吗？

之前的解读说“JSON 解析占了一半：单轮 1 次、多轮 teacher 2 次、自由对话 1 次”。这是按**失败次数**数的。换一个角度：这些失败来自多少条**不同的**模型输出？

In [35]:
json_fail = []
for name, (s, r) in runs.items():
    for _, row in r[r["error"].fillna("").str.contains("Invalid json")].iterrows():
        raw = (
            row["error"]
            .split("Invalid json output:")[1]
            .split("For troubleshooting")[0]
            .strip()
        )
        json_fail.append(
            {
                "评测": name,
                "题": f"{row['case_id']}#{row['turn'] if 'turn' in row else row['variant']}",
                "模型原始输出": " ".join(raw.split())[:60],
            }
        )
jf = pd.DataFrame(json_fail)
display(jf)
print("不同的原始输出：", jf["模型原始输出"].nunique())

,评测,题,模型原始输出
0,baseline,relative_time_05#paraphrase_1,[ “销售额”， “销售额同比增长率”， “同比增长率”， “同比增速”， “同比增幅”， “销售收入”， “销售金额”
1,mt_teacher_phase1,mt_result_ref_03#1,[ “支付订单数” ]
2,mt_free_phase1,mt_result_ref_03#1,[ “支付订单数” ]
3,mt_teacher,mt_result_ref_03#1,[ “支付订单数” ]
4,mt_teacher,mt_clarify_02#2,[ “支付金额” ]
5,mt_free,mt_result_ref_03#1,[ “支付订单数” ]


不同的原始输出： 3


<details><summary>分析</summary>

6 次失败只来自 **3 条不同的模型输出**。`mt_result_ref_03` 第 1 轮的 `[“支付订单数”]` 是同一条录制回答，在 4 次多轮评测里被回放了 4 次。

所以更准确的说法是：
- 在**发生频率**上：缓存里 946 条应输出 JSON 的回答，有 3 条带全角引号，约 0.3%。每道题有 4～5 次 JSON 调用，折算下来每道题约 1～1.5% 的概率因此中断
- 在**影响**上：它一旦发生，整条流程中断，没有任何降级。这是它值得优先修的原因：**低频但致命、修复成本低、验证成本低**（用缓存回放就能确认那 3 条输出被正确解析）

两种说法都对，但“一半”会让人高估它的频率。**数失败时要问：这些失败是独立发生的吗？**

</details>

### 10.5 一个会随日期变化的分数

`relative_time_02` 换问法“上月成交了多少单”，模型 SQL 用了 `CURRENT_DATE`。题目注入的“今天”是 2025-12-31，评测是 2026-11 跑的。

🤔 **先想一想**：如果同一份录制的回答，在 2026 年 1 月 15 日用缓存回放打分，`relative_time_02` 会被判对吗？

我们可以直接做这个实验：把 SQL 里的 `CURDATE()` / `CURRENT_DATE` 换成题目的“今天”，在数仓上执行，与标准答案比较。

In [36]:
import re

DW = yaml.safe_load((ROOT / "conf/app_config.yaml").read_text(encoding="utf-8"))[
    "db_dw"
]


def run_dw(sql: str) -> list[dict]:
    import mysql.connector

    conn = mysql.connector.connect(
        host=DW["host"],
        port=DW["port"],
        user=DW["user"],
        password=DW["password"],
        database=DW["database"],
    )
    try:
        cur = conn.cursor(dictionary=True)
        cur.execute(sql)
        return cur.fetchall()
    finally:
        conn.close()


def pin_today(sql: str, today: str) -> str:
    # 把“当前日期”函数替换成固定日期，模拟“在题目设定的那一天运行”
    return re.sub(
        r"CURDATE\(\)|CURRENT_DATE(\(\))?", f"DATE('{today}')", sql, flags=re.I
    )


try:
    for case in ["relative_time_02", "relative_time_04"]:
        row = base_r[
            (base_r["case_id"] == case) & (base_r["variant"] == "paraphrase_1")
        ].iloc[0]
        gold = run_dw(trap_by_id[case]["gold_sql"])
        as_run = run_dw(row["sql"])
        pinned = run_dw(pin_today(row["sql"], "2025-12-31"))
        print(f"{case}：{row['question']}")
        print(
            f"  标准答案 {len(gold)} 行；按今天执行 {len(as_run)} 行 → {results_equal(as_run, gold)}；"
            f"把“今天”固定为 2025-12-31 → {results_equal(pinned, gold)}"
        )
except Exception as e:
    print("连不上数仓，跳过：", type(e).__name__, e)

relative_time_02：上月成交了多少单
  标准答案 1 行；按今天执行 1 行 → False；把“今天”固定为 2025-12-31 → True
relative_time_04：过去一周每天卖了多少钱
  标准答案 7 行；按今天执行 0 行 → False；把“今天”固定为 2025-12-31 → True


<details><summary>分析</summary>

把“今天”固定之后，模型的 SQL 就是对的。也就是说：

1. 模型在这两题上**理解对了问题、写对了逻辑**，错在没有使用注入的日期。这个错误在真实使用中是真实的（用户在 2025-12-31 问“上月”，期望的是 11 月）；但它的严重程度和“口径理解错了”完全不同
2. 更隐蔽的是：**评测分数依赖于跑评测的日期**。2026 年 1 月回放，`CURDATE()` 指向 2026-01，“上月”是 2025-12，又会得到另一个答案（仍然是错的，但如果某道题恰好碰上，就可能被判对）。一个好的评测，结果只应该依赖于“被测系统 + 评测集”，不应依赖于墙上的时钟
3. 修复应该在两层：**系统层**，语义检查禁止 `CURDATE()` / `NOW()`，交给修正节点改成具体日期（路线图第 2 项）；**评测层**，执行预测 SQL 前把会话的当前日期固定为题目的 `today`（例如 MySQL 的 `SET TIMESTAMP`），让分数可复现，同时把“用了当前日期函数”单独记录为一种失败原因

</details>

### 10.6 消融实验“没有变化”说明了什么

In [37]:
q = base_r[base_r["variant"] == "question"].set_index("case_id")["passed"]
ablation = pd.DataFrame({"基线（原问法）": q})
for name in ("no_examples", "no_lint"):
    ablation[name] = runs[name][1].set_index("case_id")["passed"]
print("三组结论不同的题：", ablation[ablation.nunique(axis=1) > 1].index.tolist())
print("基线中被语义检查拦下修正过的作答：")
base_r[base_r["correct_attempts"] > 0][
    ["case_id", "variant", "correct_attempts", "passed", "lint_issues"]
]

三组结论不同的题： []
基线中被语义检查拦下修正过的作答：


,case_id,variant,correct_attempts,passed,lint_issues
24,refund_03,question,2,True,[SUM(i.pay_amount) 聚合的是 fact_order_item 的字段，但同一层查询还关联了引用它的 fact_refund，fact_order_item 的每一行会被重复计...
25,refund_03,paraphrase_1,2,True,[]
28,refund_05,question,1,True,[]
36,date_role_04,question,1,True,[]
37,date_role_04,paraphrase_1,1,True,[]
94,funnel_03,question,1,True,[]


<details><summary>分析</summary>

三组结论完全一样。“关掉语义检查分数不变”有两种可能的解释：(a) 语义检查没用；(b) 评测集测不出它的作用。怎么区分？

看被拦下修正过的作答：`refund_03`、`refund_05`、`date_role_04`、`funnel_03`。语义检查报的是“退款表关联订单明细后对明细金额求和会扇出”。关掉检查后这些题的 SQL 未经修正，结果仍然对——因为**这份数据里每条订单明细最多只有一条退款**，一对多的关联在数据上从来不会真的放大金额。

还有一个细节：`refund_03` 原问法修正了 2 次（用完了修正次数），最终 SQL 仍然带着语义检查的提示，但判分是对的。也就是说在这份数据上，语义检查对这条 SQL 的告警**没有造成实际错误**。这既可以理解为“风险真实存在、只是数据没覆盖”，也说明语义检查会让模型花两次额外的 LLM 调用去修一个在当前数据上不影响结果的问题。

所以答案是 (b)：**评测集没有覆盖某种情况，消融就测不出对应模块的价值**。改进方法是补数据（一条明细多次部分退款）和补题，这正是路线图第 4 项。这条推理很通用：一个“没有效果”的消融结果，首先要怀疑的是测量工具的覆盖范围。

</details>

### 10.7 多轮：92% → 100% 的改进有多大把握？

In [38]:
first20 = [i["id"] for i in multiturn[:20]]
mt = pd.DataFrame(
    {
        name: runs[name][1]
        .query("turn > 1 and case_id in @first20")
        .set_index(["case_id", "turn"])["passed"]
        for name in ("mt_teacher_phase1", "mt_teacher", "mt_free_phase1", "mt_free")
    }
)
print(mt.sum().to_string(), "\n（共", len(mt), "个追问轮）")
mt[mt.nunique(axis=1) > 1]

mt_teacher_phase1    22
mt_teacher           24
mt_free_phase1       21
mt_free              24 
（共 24 个追问轮）


,,mt_teacher_phase1,mt_teacher,mt_free_phase1,mt_free
case_id,turn,,,,
mt_add_filter_01,2,True,True,False,True
mt_change_dim_01,3,False,True,False,True
mt_correction_02,2,False,True,False,True


🤔 **先想一想**：teacher 92% → 100%、free 88% → 100%，各自是 2 个、3 个追问轮由错变对。第一阶段的失败原因和修复手段在 `eval_tutorial.ipynb` 第 10 节都一一对上了。这个改进的结论站得住吗？有没有需要打折扣的地方？

<details><summary>分析</summary>

站得住的部分：每一个由错变对的轮次都能对上具体的原因和具体的改动（参考 SQL、改写规则），这比一个总分的变化有说服力得多。**可解释的改进比数字上的改进更可信。**

需要打折扣的地方：

1. **样本很小**：24 个追问轮，100% 的 95% 置信区间下限是 86%（见 10.2 的计算）。“100%”不代表追问能力已经没有问题，只代表这 24 轮没有出错
2. **在测试集上调参**：第一阶段的 `mt_change_dim_01` 第 3 轮（“再按平台拆”丢了按月维度）是看了评测失败之后，在改写 Prompt 里加了“叠加维度和替换维度”的规则修好的。规则写得很通用、few-shot 也刻意用了 2024 年和不同的实体，但它的**动机**来自这道题。用同一道题来证明修好了，结论天然偏乐观
3. **两次运行的条件不完全相同**：phase1 主要是回放（未命中 1%），新结果约 25% 是真实调用；评测集也加了 2 段对话和 `gold_answer_contains`。比较“同一批 20 段对话的追问轮”已经尽量控制了变量，但不是严格的对照

应对第 2 点的标准做法是**留出集（held-out set）**：把评测题分成“开发集”（看失败、改 Prompt 用）和“测试集”（只在最后评估、不看具体失败）。题目少的时候很难分，一个折中是：每次根据某道题的失败改了系统，就**新写几道同类型、不同表述的题**来验证，而不是只看原来那道题是否变对。

</details>

### 10.8 意图准确率 92%：都是模型的错吗？

In [39]:
cols = [
    "case_id",
    "turn",
    "gold_intent",
    "intent",
    "passed",
    "stage",
    "clarify_question",
]
for name in ("mt_teacher", "mt_free"):
    r = runs[name][1]
    wrong = r[(r["turn"] > 1) & (r["intent"] != r["gold_intent"])]
    print(f"== {name}")
    display(wrong[cols])

== mt_teacher


,case_id,turn,gold_intent,intent,passed,stage,clarify_question
7,mt_ellipsis_03,3,followup,clarify,True,通过,你问的是哪个品牌的件单价？
45,mt_clarify_01,2,clarify,followup,False,意图错误,NaN


== mt_free


,case_id,turn,gold_intent,intent,passed,stage,clarify_question
33,mt_result_ref_03,2,result_qa,followup,True,通过,NaN
45,mt_clarify_01,2,clarify,followup,False,意图错误,NaN


<details><summary>分析</summary>

4 次意图“错误”，性质各不相同：

| 情况 | 意图 | 结果 | 是不是问题 |
| --- | --- | --- | --- |
| teacher `mt_ellipsis_03` 第 3 轮“它的件单价” | followup → clarify | 对（被默认回答救回来） | **是**：不该问却问了，打扰了用户 |
| 两种模式 `mt_clarify_01` 第 2 轮“那最近的呢” | clarify → followup | 错 | **是**：该问没问 |
| free `mt_result_ref_03` 第 2 轮“哪种最少” | result_qa → followup | 对 | **不是**：第 1 轮查询失败了，没有上一轮结果可用，模型改为重新查库，这正是设计好的退路 |

第三种情况说明**指标定义有缺陷**：`gold_intent` 是按“理想的对话”标注的，但自由对话里前一轮失败后，“正确的意图”已经变了。意图准确率在 free 模式下把一次正确的行为算成了错误。更合理的做法是：free 模式下当上一轮没有结果时，result_qa 和 followup 都视为正确；或者干脆只在 teacher 模式下统计意图准确率。

这提醒我们：**一个指标在设计时的假设，要在它被使用的每个场景下重新检验**。

另外，反问相关的样本一共只有 2 段需要反问的对话 + 若干不该反问的轮次，“反问只成功 1 次”这个结论几乎没有统计意义。路线图第 3 项“多准备一些需要反问的评测题”是做反问优化的前提：**没有足够的测量，就无法判断改动是否有效**。

</details>

## 11. 问题清单与练习

### 11.1 精读中发现的问题

按“对评测结论的影响”排序。**评测代码的问题**会让分数失真；**解读的问题**是读分数时需要打的折扣。

| # | 类型 | 问题 | 复现 | 影响 | 修复思路 |
| --- | --- | --- | --- | --- | --- |
| 1 | 评测代码 | `compare_with_baseline` 把“本次有必过题失败”当退化，而基线里 `activity_01` 本来就失败：基线和自己比也判退化，每次评测退出码都是 1 | 6.3 | 门禁永远报警，等于没有门禁 | 只把新增的必过题失败当退化理由 |
| 1b | 评测代码 | `compare_with_baseline` 不检查两次评测的题集是否一致，用不同题集的整体准确率比较 | 6.3 | 门禁可能放过真实退化（只跑原问法时有约 5 个点的虚假余量） | 只在 `per_case` 交集上算准确率，题集不同时报错 |
| 2 | 评测代码 | 预测 SQL 里的 `CURDATE()` / `NOW()` 在评测时按真实日期执行，分数依赖跑评测的日期 | 10.5 | 缓存回放也不可复现 | 执行前 `SET TIMESTAMP` 固定为题目的 `today`，并单独记录“使用了当前日期函数” |
| 3 | 评测代码 | ×100 倍数规则对所有列生效（计数、金额） | 5.4 A、B | 少见，但会误放 | 只对标准值都在 [0, 1] 的列允许 ×100 |
| 4 | 评测代码 | 多重集合比较用贪心匹配，判定结果可能随行顺序变化 | 5.4 E、F | 很少见 | 改用二分图匹配；加“打乱行顺序判定不变”的性质测试 |
| 5 | 评测代码 | 直接回答只做子串匹配，不检查说法和数字 | 9.3 | 目前只有 2 轮，影响小 | 标注关键数字，或要求结构化回答 |
| 6 | 指标定义 | free 模式的意图准确率把“前一轮失败后改为查库”算成意图错误 | 10.8 | 低估意图准确率 | free 模式下按实际状态判断，或只在 teacher 模式统计 |
| 7 | 评测集 | `expect` 把某一种写法需要的表当成必需（`order_vs_pay_02` 的 `dim_date`） | 10.3 / 6.2 | 归因误导 | `expect` 只写所有正确写法都必需的表 |
| 8 | 评测集 | 换问法存在歧义（`activity_01`“参加活动的商品销量”可理解为按商品拆分） | 10.3 | 低估换问法准确率 | 换问法也要审核“唯一理解” |
| 9 | 评测集 | 数据里每条明细最多一次退款，语义检查的价值测不出 | 10.6 | 消融结论无效 | 生成器补“一条明细多次部分退款”，并出题（路线图第 4 项） |
| 10 | 解读 | 回放的失败不是独立样本（6 次 JSON 失败来自 3 条输出） | 10.4 | 高估频率 | 统计不同输出数，或按真实调用的运行统计 |
| 11 | 解读 | 样本小：单轮 1 题 = 2 个点，多轮 24 个追问轮；2 个点的门禁阈值与噪声同量级 | 10.2 | 容易把噪声当改进 | 报告置信区间；配对比较；扩充题目 |
| 12 | 解读 | 在评测集上看失败、改 Prompt、再用同一批题验证 | 10.7 | 结论偏乐观 | 每次针对性修复都补同类新题验证 |

### 11.2 这些问题对路线图的意义

- 路线图第 1 项（JSON 容错）：10.4 节支持优先做它，理由要改成“低频但致命、修复成本低”，而不是“占一半”
- 路线图第 2 项（禁止 `CURDATE()`）：10.5 节说明它同时是系统问题和评测问题，建议评测层的问题 #2 一起修
- 路线图第 3 项（反问）：10.8 节说明在动 Prompt 之前，**先补评测题**，否则无法判断改动是否有效
- 路线图第 4 项（评测覆盖）：10.6 节给出了“为什么必须补数据”的完整论证
- 评测代码的问题 #1、#1b、#3、#4 都是 `compare.py` / `scoring.py` 里的纯函数，修起来成本低、可以用单元测试锁住，适合作为练手

### 11.3 练习

由易到难。每个练习都只需要改纯函数模块，写完用 `uv run pytest tests/test_compare.py tests/test_scoring.py` 验证。

1. **读**：`summarize_dialogs` 里的 `must_pass_failed` 和单轮的 `summarize` 有什么不同？为什么多轮按“整段对话”判断必过题？
2. **写测试**：为 `results_equal` 写一个性质测试：对 `tests/test_compare.py` 里每个判对的例子，把预测结果的行随机打乱 20 次，判定都应为真。运行它，你应该能复现问题 #4
3. **修 bug**：把 `_column_matches` 的贪心匹配改成二分图匹配（提示：对每个标准值，用 DFS 找增广路径，共 20 行左右），让练习 2 的测试通过，且原有测试全部通过
4. **收紧规则**：按问题 #3 的思路限制 ×100 规则。改完后按 CLAUDE.md 的要求，跑 `check_dataset` 和 `--predictor gold`、`--predictor trap`，确认没有把原本判对的标准答案判错
5. **改门禁**：让 `compare_with_baseline` 满足“基线和自己比不退化”，并且只在两次评测共同的题上比较准确率；在 `tests/test_scoring.py` 里加用例复现 6.3 节的两个场景
6. **设计**：为问题 #2 设计方案：在 `runner.execute` 里怎么固定“当前日期”？`SET TIMESTAMP` 会影响哪些函数（`NOW()`、`CURDATE()`、`UNIX_TIMESTAMP()`）？连接池里的连接复用会不会让这个设置“泄漏”到下一个请求？

### 11.4 一张可以带走的检查表

下次读任何一份评测报告时，可以按这个顺序问自己：

1. **条件**：用的哪个模型、哪份评测集、哪些开关？是真实调用还是回放？
2. **构成**：总分由哪些群体组成？分开看各是多少？
3. **不确定性**：样本多大？差异有几道题？置信区间多宽？是否配对？
4. **逐题**：每个失败的 SQL 长什么样？按根因分组后是什么分布？
5. **独立性**：这些失败是独立发生的，还是同一个原因 / 同一条录制输出的重复？
6. **度量本身**：判分器、归因、指标定义在这个场景下的假设成立吗？有没有判错的可能？
7. **对比的公平性**：比较的两组在题集、数据、提示词、调用方式上是否一致？改进是不是在同一批题上调出来的？